In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

for file in drive_root.glob("stage_2C*.csv"):
    print(file)

/content/drive/MyDrive/stage_2C_operating_condition_sweep_temperature_summary_time_results.csv


In [ ]:
from pathlib import Path
import shutil

source_file = Path("/content/drive/MyDrive/stage_2C_operating_condition_sweep_temperature_summary_time_results.csv")

target_folder = Path("dataset/comsol/processed")
target_folder.mkdir(parents=True, exist_ok=True)

target_file = target_folder / "stage_2C_operating_condition_sweep_temperature_summary_time_results.csv"

shutil.copy(source_file, target_file)

print("Copied file to:", target_file)
print("File exists:", target_file.exists())

Copied file to: dataset/comsol/processed/stage_2C_operating_condition_sweep_temperature_summary_time_results.csv
File exists: True


In [ ]:
"""
Stage 3A Dataset Inspection Script
Project: FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction
Author: Frank Ouma
Contact: +254725582132

Purpose:
This script inspects the merged Stage 2C COMSOL operating-condition sweep dataset
before surrogate model training. It checks data quality, verifies expected physical
trends, creates summary statistics, and saves inspection plots and reports.

Expected input file:
    dataset/comsol/processed/stage_2C_operating_condition_sweep_temperature_summary_time_results.csv

Generated outputs:
    reports/stage_3A_dataset_inspection_report.txt
    dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv
    figures/stage_3A_dataset_inspection/Tmax_vs_time_by_current.png
    figures/stage_3A_dataset_inspection/final_Tmax_vs_current.png
    figures/stage_3A_dataset_inspection/final_Tmax_vs_hconv.png
    figures/stage_3A_dataset_inspection/final_Tmax_vs_Tamb.png
    figures/stage_3A_dataset_inspection/final_thermal_gradient_vs_current.png
"""

from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt


PROJECT_NAME = "FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction"
AUTHOR = "Frank Ouma"
CONTACT = "+254725582132"


REQUIRED_COLUMNS = [
    "I_pack_A",
    "h_conv_W_per_m2K",
    "Tamb_K",
    "Tamb_C",
    "time_s",
    "Tmin_K",
    "Tavg_K",
    "Tmax_K",
    "Tmin_C",
    "Tavg_C",
    "Tmax_C",
    "thermal_gradient_K",
    "thermal_gradient_C",
]


def find_project_root() -> Path:
    """Return the project root based on the current working directory."""
    current = Path.cwd().resolve()

    if (current / "dataset").exists():
        return current

    for parent in current.parents:
        if (parent / "dataset").exists():
            return parent

    return current


def load_dataset(csv_path: Path) -> pd.DataFrame:
    """Load the Stage 2C merged COMSOL dataset."""
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {csv_path}\n"
            "Place the merged Stage 2C CSV in dataset/comsol/processed/ "
            "or update the csv_path variable in main()."
        )

    df = pd.read_csv(csv_path)
    return df


def validate_required_columns(df: pd.DataFrame) -> None:
    """Check that all required columns exist."""
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]

    if missing:
        raise ValueError(
            "The dataset is missing required columns:\n" + "\n".join(missing)
        )


def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and standardize the dataset for inspection and later modeling."""
    cleaned = df.copy()

    numeric_columns = [
        "I_pack_A",
        "h_conv_W_per_m2K",
        "Tamb_K",
        "Tamb_C",
        "time_s",
        "Tmin_K",
        "Tavg_K",
        "Tmax_K",
        "Tmin_C",
        "Tavg_C",
        "Tmax_C",
        "thermal_gradient_K",
        "thermal_gradient_C",
    ]

    for col in numeric_columns:
        cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

    cleaned = cleaned.sort_values(
        by=["I_pack_A", "h_conv_W_per_m2K", "Tamb_C", "time_s"]
    ).reset_index(drop=True)

    return cleaned


def create_summary_statistics(df: pd.DataFrame) -> dict:
    """Create core dataset statistics."""
    stats = {
        "row_count": len(df),
        "column_count": len(df.columns),
        "missing_values_total": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "current_values": sorted(df["I_pack_A"].dropna().unique().tolist()),
        "cooling_values": sorted(df["h_conv_W_per_m2K"].dropna().unique().tolist()),
        "ambient_values_C": sorted(df["Tamb_C"].dropna().unique().tolist()),
        "time_min_s": float(df["time_s"].min()),
        "time_max_s": float(df["time_s"].max()),
        "Tmin_C_min": float(df["Tmin_C"].min()),
        "Tmin_C_max": float(df["Tmin_C"].max()),
        "Tavg_C_min": float(df["Tavg_C"].min()),
        "Tavg_C_max": float(df["Tavg_C"].max()),
        "Tmax_C_min": float(df["Tmax_C"].min()),
        "Tmax_C_max": float(df["Tmax_C"].max()),
        "thermal_gradient_C_min": float(df["thermal_gradient_C"].min()),
        "thermal_gradient_C_max": float(df["thermal_gradient_C"].max()),
    }
    return stats


def extract_final_time_cases(df: pd.DataFrame) -> pd.DataFrame:
    """Extract final-time rows for each operating condition."""
    idx = df.groupby(["I_pack_A", "h_conv_W_per_m2K", "Tamb_C"])["time_s"].idxmax()
    final_df = df.loc[idx].copy()
    final_df = final_df.sort_values(
        by=["I_pack_A", "h_conv_W_per_m2K", "Tamb_C"]
    ).reset_index(drop=True)
    return final_df


def check_physical_consistency(final_df: pd.DataFrame) -> list[str]:
    """Run simple physical consistency checks on final-time results."""
    findings = []

    for h_conv in sorted(final_df["h_conv_W_per_m2K"].unique()):
        for tamb in sorted(final_df["Tamb_C"].unique()):
            subset = final_df[
                (final_df["h_conv_W_per_m2K"] == h_conv)
                & (final_df["Tamb_C"] == tamb)
            ].sort_values("I_pack_A")

            if len(subset) > 1:
                is_monotonic = subset["Tmax_C"].is_monotonic_increasing
                if is_monotonic:
                    findings.append(
                        f"PASS: Tmax increases with I_pack at h_conv={h_conv}, Tamb={tamb} degC."
                    )
                else:
                    findings.append(
                        f"CHECK: Tmax does not strictly increase with I_pack at h_conv={h_conv}, Tamb={tamb} degC."
                    )

    for current in sorted(final_df["I_pack_A"].unique()):
        for tamb in sorted(final_df["Tamb_C"].unique()):
            subset = final_df[
                (final_df["I_pack_A"] == current)
                & (final_df["Tamb_C"] == tamb)
            ].sort_values("h_conv_W_per_m2K")

            if len(subset) > 1:
                is_monotonic = subset["Tmax_C"].is_monotonic_decreasing
                if is_monotonic:
                    findings.append(
                        f"PASS: Tmax decreases with h_conv at I_pack={current}, Tamb={tamb} degC."
                    )
                else:
                    findings.append(
                        f"CHECK: Tmax does not strictly decrease with h_conv at I_pack={current}, Tamb={tamb} degC."
                    )

    for current in sorted(final_df["I_pack_A"].unique()):
        for h_conv in sorted(final_df["h_conv_W_per_m2K"].unique()):
            subset = final_df[
                (final_df["I_pack_A"] == current)
                & (final_df["h_conv_W_per_m2K"] == h_conv)
            ].sort_values("Tamb_C")

            if len(subset) > 1:
                is_monotonic = subset["Tmax_C"].is_monotonic_increasing
                if is_monotonic:
                    findings.append(
                        f"PASS: Tmax increases with Tamb at I_pack={current}, h_conv={h_conv}."
                    )
                else:
                    findings.append(
                        f"CHECK: Tmax does not strictly increase with Tamb at I_pack={current}, h_conv={h_conv}."
                    )

    return findings


def save_report(
    report_path: Path,
    stats: dict,
    final_df: pd.DataFrame,
    consistency_findings: list[str],
    csv_path: Path,
    cleaned_csv_path: Path,
) -> None:
    """Write the dataset inspection report."""
    with report_path.open("w", encoding="utf-8") as f:
        f.write("Stage 3A Dataset Inspection Report\n")
        f.write(f"Project: {PROJECT_NAME}\n")
        f.write(f"Author: {AUTHOR}\n")
        f.write(f"Contact: {CONTACT}\n")
        f.write("\n")

        f.write("Input Dataset\n")
        f.write(f"Source file: {csv_path}\n")
        f.write(f"Cleaned file: {cleaned_csv_path}\n")
        f.write("\n")

        f.write("Dataset Size\n")
        f.write(f"Rows: {stats['row_count']}\n")
        f.write(f"Columns: {stats['column_count']}\n")
        f.write(f"Missing values: {stats['missing_values_total']}\n")
        f.write(f"Duplicate rows: {stats['duplicate_rows']}\n")
        f.write("\n")

        f.write("Input Ranges\n")
        f.write(f"I_pack values A: {stats['current_values']}\n")
        f.write(f"h_conv values W per m2K: {stats['cooling_values']}\n")
        f.write(f"Tamb values degC: {stats['ambient_values_C']}\n")
        f.write(f"Time range s: {stats['time_min_s']} to {stats['time_max_s']}\n")
        f.write("\n")

        f.write("Output Ranges\n")
        f.write(f"Tmin range degC: {stats['Tmin_C_min']:.4f} to {stats['Tmin_C_max']:.4f}\n")
        f.write(f"Tavg range degC: {stats['Tavg_C_min']:.4f} to {stats['Tavg_C_max']:.4f}\n")
        f.write(f"Tmax range degC: {stats['Tmax_C_min']:.4f} to {stats['Tmax_C_max']:.4f}\n")
        f.write(f"Thermal gradient range degC: {stats['thermal_gradient_C_min']:.4f} to {stats['thermal_gradient_C_max']:.4f}\n")
        f.write("\n")

        f.write("Final-Time Cases\n")
        f.write(final_df.to_string(index=False))
        f.write("\n\n")

        f.write("Physical Consistency Checks\n")
        for item in consistency_findings:
            f.write(f"{item}\n")
        f.write("\n")

        f.write("Inspection Conclusion\n")
        f.write("The dataset contains the expected COMSOL input-output structure for surrogate model preparation. ")
        f.write("The main input features are I_pack_A, h_conv_W_per_m2K, Tamb_C, and time_s. ")
        f.write("The primary prediction target for the first surrogate model should be Tmax_C, with Tavg_C and thermal_gradient_C as secondary targets.\n")


def plot_tmax_vs_time_by_current(df: pd.DataFrame, figure_dir: Path) -> None:
    """Plot Tmax over time for each current at baseline Tamb and h_conv."""
    baseline_h = sorted(df["h_conv_W_per_m2K"].unique())[0]
    baseline_tamb = sorted(df["Tamb_C"].unique())[1] if len(df["Tamb_C"].unique()) > 1 else sorted(df["Tamb_C"].unique())[0]

    subset = df[
        (df["h_conv_W_per_m2K"] == baseline_h)
        & (df["Tamb_C"] == baseline_tamb)
    ]

    plt.figure(figsize=(10, 6))
    for current, group in subset.groupby("I_pack_A"):
        group = group.sort_values("time_s")
        plt.plot(group["time_s"], group["Tmax_C"], linewidth=2, label=f"I_pack={current:g} A")

    plt.xlabel("Time (s)")
    plt.ylabel("Maximum Cell Temperature (degC)")
    plt.title(f"Tmax vs Time by Current, h_conv={baseline_h:g} W/m2K, Tamb={baseline_tamb:g} degC")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(figure_dir / "Tmax_vs_time_by_current.png", dpi=200)
    plt.close()


def plot_final_tmax_vs_current(final_df: pd.DataFrame, figure_dir: Path) -> None:
    """Plot final Tmax against current."""
    summary = final_df.groupby("I_pack_A", as_index=False)["Tmax_C"].mean()

    plt.figure(figsize=(8, 5))
    plt.plot(summary["I_pack_A"], summary["Tmax_C"], marker="o", linewidth=2)
    plt.xlabel("I_pack (A)")
    plt.ylabel("Mean Final Tmax (degC)")
    plt.title("Final Maximum Temperature vs Battery Current")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_dir / "final_Tmax_vs_current.png", dpi=200)
    plt.close()


def plot_final_tmax_vs_hconv(final_df: pd.DataFrame, figure_dir: Path) -> None:
    """Plot final Tmax against cooling coefficient."""
    summary = final_df.groupby("h_conv_W_per_m2K", as_index=False)["Tmax_C"].mean()

    plt.figure(figsize=(8, 5))
    plt.plot(summary["h_conv_W_per_m2K"], summary["Tmax_C"], marker="o", linewidth=2)
    plt.xlabel("h_conv (W/m2K)")
    plt.ylabel("Mean Final Tmax (degC)")
    plt.title("Final Maximum Temperature vs Cooling Strength")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_dir / "final_Tmax_vs_hconv.png", dpi=200)
    plt.close()


def plot_final_tmax_vs_tamb(final_df: pd.DataFrame, figure_dir: Path) -> None:
    """Plot final Tmax against ambient temperature."""
    summary = final_df.groupby("Tamb_C", as_index=False)["Tmax_C"].mean()

    plt.figure(figsize=(8, 5))
    plt.plot(summary["Tamb_C"], summary["Tmax_C"], marker="o", linewidth=2)
    plt.xlabel("Ambient Temperature (degC)")
    plt.ylabel("Mean Final Tmax (degC)")
    plt.title("Final Maximum Temperature vs Ambient Temperature")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_dir / "final_Tmax_vs_Tamb.png", dpi=200)
    plt.close()


def plot_final_gradient_vs_current(final_df: pd.DataFrame, figure_dir: Path) -> None:
    """Plot final thermal gradient against current."""
    summary = final_df.groupby("I_pack_A", as_index=False)["thermal_gradient_C"].mean()

    plt.figure(figsize=(8, 5))
    plt.plot(summary["I_pack_A"], summary["thermal_gradient_C"], marker="o", linewidth=2)
    plt.xlabel("I_pack (A)")
    plt.ylabel("Mean Final Thermal Gradient (degC)")
    plt.title("Final Thermal Gradient vs Battery Current")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_dir / "final_thermal_gradient_vs_current.png", dpi=200)
    plt.close()


def main() -> None:
    project_root = find_project_root()

    csv_path = project_root / "dataset" / "comsol" / "processed" / "stage_2C_operating_condition_sweep_temperature_summary_time_results.csv"
    cleaned_csv_path = project_root / "dataset" / "comsol" / "processed" / "stage_2C_cleaned_for_surrogate.csv"
    report_dir = project_root / "reports"
    figure_dir = project_root / "figures" / "stage_3A_dataset_inspection"

    report_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    cleaned_csv_path.parent.mkdir(parents=True, exist_ok=True)

    print("Stage 3A Dataset Inspection")
    print(f"Project root: {project_root}")
    print(f"Reading dataset: {csv_path}")

    df = load_dataset(csv_path)
    validate_required_columns(df)
    cleaned = clean_dataset(df)
    cleaned.to_csv(cleaned_csv_path, index=False)

    stats = create_summary_statistics(cleaned)
    final_df = extract_final_time_cases(cleaned)
    consistency_findings = check_physical_consistency(final_df)

    report_path = report_dir / "stage_3A_dataset_inspection_report.txt"
    save_report(
        report_path=report_path,
        stats=stats,
        final_df=final_df,
        consistency_findings=consistency_findings,
        csv_path=csv_path,
        cleaned_csv_path=cleaned_csv_path,
    )

    plot_tmax_vs_time_by_current(cleaned, figure_dir)
    plot_final_tmax_vs_current(final_df, figure_dir)
    plot_final_tmax_vs_hconv(final_df, figure_dir)
    plot_final_tmax_vs_tamb(final_df, figure_dir)
    plot_final_gradient_vs_current(final_df, figure_dir)

    print("Inspection completed successfully.")
    print(f"Cleaned dataset: {cleaned_csv_path}")
    print(f"Inspection report: {report_path}")
    print(f"Figures folder: {figure_dir}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print("Execution failed.")
        print(str(exc))
        sys.exit(1)


Stage 3A Dataset Inspection
Project root: /content
Reading dataset: /content/dataset/comsol/processed/stage_2C_operating_condition_sweep_temperature_summary_time_results.csv
Inspection completed successfully.
Cleaned dataset: /content/dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv
Inspection report: /content/reports/stage_3A_dataset_inspection_report.txt
Figures folder: /content/figures/stage_3A_dataset_inspection


In [ ]:
from google.colab import files

files.download("dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path

file_path = Path("dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv")
print("FOUND" if file_path.exists() else "MISSING")
print(file_path)

FOUND
dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv


In [ ]:
"""
Stage 3B Surrogate Model Training Script
Project: FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction
Author: Frank Ouma
Contact: +254725582132

Purpose:
This script trains the first surrogate model using the cleaned Stage 2C COMSOL dataset.
The model learns to predict maximum battery cell temperature from operating inputs.

Input dataset:
    dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv

Inputs used for training:
    I_pack_A
    h_conv_W_per_m2K
    Tamb_C
    time_s

Prediction target:
    Tmax_C

Generated outputs:
    models/stage_3B_tmax_random_forest_model.pkl
    reports/stage_3B_surrogate_training_report.txt
    figures/stage_3B_surrogate_training/predicted_vs_actual_Tmax.png
    figures/stage_3B_surrogate_training/error_distribution_Tmax.png
    figures/stage_3B_surrogate_training/feature_importance_Tmax.png
    dataset/comsol/processed/stage_3B_train_test_predictions.csv
"""

from pathlib import Path
import sys
import pickle

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


PROJECT_NAME = "FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction"
AUTHOR = "Frank Ouma"
CONTACT = "+254725582132"

FEATURE_COLUMNS = [
    "I_pack_A",
    "h_conv_W_per_m2K",
    "Tamb_C",
    "time_s",
]

TARGET_COLUMN = "Tmax_C"


def find_project_root() -> Path:
    """Return project root based on current working directory."""
    current = Path.cwd().resolve()

    if (current / "dataset").exists():
        return current

    for parent in current.parents:
        if (parent / "dataset").exists():
            return parent

    return current


def load_training_dataset(csv_path: Path) -> pd.DataFrame:
    """Load cleaned Stage 2C dataset."""
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Training dataset not found: {csv_path}\n"
            "Expected file: dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv"
        )

    df = pd.read_csv(csv_path)
    return df


def validate_dataset(df: pd.DataFrame) -> None:
    """Check that all required training columns exist and are usable."""
    required = FEATURE_COLUMNS + [TARGET_COLUMN]
    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError("Missing required columns: " + ", ".join(missing))

    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    missing_values = df[required].isna().sum()
    if missing_values.sum() > 0:
        raise ValueError(
            "Missing or non-numeric values found in training columns:\n"
            + missing_values.to_string()
        )


def prepare_training_data(df: pd.DataFrame):
    """Prepare feature matrix and target vector."""
    X = df[FEATURE_COLUMNS].copy()
    y = df[TARGET_COLUMN].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        shuffle=True,
    )

    return X_train, X_test, y_train, y_test


def train_random_forest(X_train: pd.DataFrame, y_train: pd.Series) -> RandomForestRegressor:
    """Train the first Random Forest Tmax surrogate model."""
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    return model


def evaluate_model(model, X_train, X_test, y_train, y_test) -> dict:
    """Evaluate model on train and test data."""
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_rmse = mean_squared_error(y_train, train_pred) ** 0.5
    test_rmse = mean_squared_error(y_test, test_pred) ** 0.5

    metrics = {
        "train_mae_C": mean_absolute_error(y_train, train_pred),
        "test_mae_C": mean_absolute_error(y_test, test_pred),
        "train_rmse_C": train_rmse,
        "test_rmse_C": test_rmse,
        "train_r2": r2_score(y_train, train_pred),
        "test_r2": r2_score(y_test, test_pred),
        "train_predictions": train_pred,
        "test_predictions": test_pred,
    }

    return metrics


def create_prediction_table(X_train, X_test, y_train, y_test, metrics) -> pd.DataFrame:
    """Create combined prediction table for train and test results."""
    train_df = X_train.copy()
    train_df["dataset_split"] = "train"
    train_df["actual_Tmax_C"] = y_train.values
    train_df["predicted_Tmax_C"] = metrics["train_predictions"]

    test_df = X_test.copy()
    test_df["dataset_split"] = "test"
    test_df["actual_Tmax_C"] = y_test.values
    test_df["predicted_Tmax_C"] = metrics["test_predictions"]

    combined = pd.concat([train_df, test_df], ignore_index=True)
    combined["prediction_error_C"] = combined["predicted_Tmax_C"] - combined["actual_Tmax_C"]
    combined["absolute_error_C"] = combined["prediction_error_C"].abs()

    return combined


def save_model(model, model_path: Path) -> None:
    """Save trained model using pickle."""
    with model_path.open("wb") as f:
        pickle.dump(
            {
                "model": model,
                "feature_columns": FEATURE_COLUMNS,
                "target_column": TARGET_COLUMN,
                "project": PROJECT_NAME,
                "author": AUTHOR,
            },
            f,
        )


def save_training_report(report_path: Path, df: pd.DataFrame, metrics: dict, model: RandomForestRegressor) -> None:
    """Save professional model training report."""
    feature_importance = pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "importance": model.feature_importances_,
        }
    ).sort_values("importance", ascending=False)

    with report_path.open("w", encoding="utf-8") as f:
        f.write("Stage 3B Surrogate Model Training Report\n")
        f.write(f"Project: {PROJECT_NAME}\n")
        f.write(f"Author: {AUTHOR}\n")
        f.write(f"Contact: {CONTACT}\n")
        f.write("\n")

        f.write("Model Objective\n")
        f.write("The objective of this stage is to train the first surrogate model for predicting maximum battery cell temperature.\n")
        f.write("The surrogate model approximates COMSOL simulation behavior using operating inputs and time.\n")
        f.write("\n")

        f.write("Training Dataset\n")
        f.write(f"Rows used: {len(df)}\n")
        f.write(f"Input features: {FEATURE_COLUMNS}\n")
        f.write(f"Prediction target: {TARGET_COLUMN}\n")
        f.write("\n")

        f.write("Model Type\n")
        f.write("RandomForestRegressor\n")
        f.write("This model is used as a baseline accuracy benchmark. It is not the final FPGA deployment model.\n")
        f.write("\n")

        f.write("Performance Metrics\n")
        f.write(f"Train MAE: {metrics['train_mae_C']:.6f} degC\n")
        f.write(f"Test MAE: {metrics['test_mae_C']:.6f} degC\n")
        f.write(f"Train RMSE: {metrics['train_rmse_C']:.6f} degC\n")
        f.write(f"Test RMSE: {metrics['test_rmse_C']:.6f} degC\n")
        f.write(f"Train R2: {metrics['train_r2']:.6f}\n")
        f.write(f"Test R2: {metrics['test_r2']:.6f}\n")
        f.write("\n")

        f.write("Feature Importance\n")
        f.write(feature_importance.to_string(index=False))
        f.write("\n\n")

        f.write("Engineering Interpretation\n")
        f.write("A low test error and high R2 score indicate that the COMSOL temperature response is learnable from the selected inputs.\n")
        f.write("The model provides a fast approximation of the COMSOL simulation output for Tmax_C.\n")
        f.write("This trained model is useful as a benchmark before moving to simpler FPGA-friendly models and quantization.\n")
        f.write("\n")

        f.write("Next Step\n")
        f.write("The next stage should train a smaller FPGA-friendly surrogate model, such as polynomial regression or a compact neural network.\n")
        f.write("That model can then be quantized to int16 or int8 for hardware-oriented inference.\n")


def plot_predicted_vs_actual(prediction_df: pd.DataFrame, figure_path: Path) -> None:
    """Plot predicted vs actual Tmax."""
    plt.figure(figsize=(7, 7))

    test_df = prediction_df[prediction_df["dataset_split"] == "test"]
    plt.scatter(test_df["actual_Tmax_C"], test_df["predicted_Tmax_C"], alpha=0.75)

    min_val = min(test_df["actual_Tmax_C"].min(), test_df["predicted_Tmax_C"].min())
    max_val = max(test_df["actual_Tmax_C"].max(), test_df["predicted_Tmax_C"].max())
    plt.plot([min_val, max_val], [min_val, max_val], linewidth=2)

    plt.xlabel("Actual Tmax (degC)")
    plt.ylabel("Predicted Tmax (degC)")
    plt.title("Stage 3B Random Forest Surrogate: Predicted vs Actual Tmax")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def plot_error_distribution(prediction_df: pd.DataFrame, figure_path: Path) -> None:
    """Plot test error distribution."""
    test_df = prediction_df[prediction_df["dataset_split"] == "test"]

    plt.figure(figsize=(8, 5))
    plt.hist(test_df["prediction_error_C"], bins=30, edgecolor="black")
    plt.xlabel("Prediction Error (degC)")
    plt.ylabel("Count")
    plt.title("Stage 3B Random Forest Surrogate: Test Error Distribution")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def plot_feature_importance(model: RandomForestRegressor, figure_path: Path) -> None:
    """Plot feature importance."""
    importance_df = pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "importance": model.feature_importances_,
        }
    ).sort_values("importance", ascending=True)

    plt.figure(figsize=(8, 5))
    plt.barh(importance_df["feature"], importance_df["importance"])
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title("Stage 3B Random Forest Surrogate: Feature Importance")
    plt.grid(True, axis="x")
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def run_single_prediction_example(model: RandomForestRegressor) -> float:
    """Run one simple example prediction."""
    example = pd.DataFrame(
        [
            {
                "I_pack_A": 40.0,
                "h_conv_W_per_m2K": 20.0,
                "Tamb_C": 30.0,
                "time_s": 600.0,
            }
        ]
    )

    predicted_tmax = float(model.predict(example)[0])
    return predicted_tmax


def main() -> None:
    project_root = find_project_root()

    dataset_path = project_root / "dataset" / "comsol" / "processed" / "stage_2C_cleaned_for_surrogate.csv"
    output_dataset_path = project_root / "dataset" / "comsol" / "processed" / "stage_3B_train_test_predictions.csv"
    model_dir = project_root / "models"
    report_dir = project_root / "reports"
    figure_dir = project_root / "figures" / "stage_3B_surrogate_training"

    model_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    output_dataset_path.parent.mkdir(parents=True, exist_ok=True)

    model_path = model_dir / "stage_3B_tmax_random_forest_model.pkl"
    report_path = report_dir / "stage_3B_surrogate_training_report.txt"

    print("Stage 3B Surrogate Model Training")
    print(f"Project root: {project_root}")
    print(f"Reading dataset: {dataset_path}")

    df = load_training_dataset(dataset_path)
    validate_dataset(df)

    X_train, X_test, y_train, y_test = prepare_training_data(df)
    model = train_random_forest(X_train, y_train)
    metrics = evaluate_model(model, X_train, X_test, y_train, y_test)

    prediction_df = create_prediction_table(X_train, X_test, y_train, y_test, metrics)
    prediction_df.to_csv(output_dataset_path, index=False)

    save_model(model, model_path)
    save_training_report(report_path, df, metrics, model)

    plot_predicted_vs_actual(prediction_df, figure_dir / "predicted_vs_actual_Tmax.png")
    plot_error_distribution(prediction_df, figure_dir / "error_distribution_Tmax.png")
    plot_feature_importance(model, figure_dir / "feature_importance_Tmax.png")

    example_prediction = run_single_prediction_example(model)

    print("Training completed successfully.")
    print(f"Train MAE: {metrics['train_mae_C']:.6f} degC")
    print(f"Test MAE: {metrics['test_mae_C']:.6f} degC")
    print(f"Train RMSE: {metrics['train_rmse_C']:.6f} degC")
    print(f"Test RMSE: {metrics['test_rmse_C']:.6f} degC")
    print(f"Train R2: {metrics['train_r2']:.6f}")
    print(f"Test R2: {metrics['test_r2']:.6f}")
    print(f"Example prediction, I=40A, h=20W/m2K, Tamb=30C, t=600s: Tmax={example_prediction:.4f} degC")
    print(f"Saved model: {model_path}")
    print(f"Saved report: {report_path}")
    print(f"Saved predictions: {output_dataset_path}")
    print(f"Saved figures: {figure_dir}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print("Execution failed.")
        print(str(exc))
        sys.exit(1)


Stage 3B Surrogate Model Training
Project root: /content
Reading dataset: /content/dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv
Training completed successfully.
Train MAE: 0.149968 degC
Test MAE: 0.399936 degC
Train RMSE: 0.245948 degC
Test RMSE: 0.642365 degC
Train R2: 0.999910
Test R2: 0.999343
Example prediction, I=40A, h=20W/m2K, Tamb=30C, t=600s: Tmax=71.7068 degC
Saved model: /content/models/stage_3B_tmax_random_forest_model.pkl
Saved report: /content/reports/stage_3B_surrogate_training_report.txt
Saved predictions: /content/dataset/comsol/processed/stage_3B_train_test_predictions.csv
Saved figures: /content/figures/stage_3B_surrogate_training


In [ ]:
from google.colab import files
import shutil

shutil.make_archive("stage_3B_surrogate_training_outputs", "zip", ".")

files.download("stage_3B_surrogate_training_outputs.zip")

OSError: [Errno 95] Operation not supported: 'drive/MyDrive/Recommend.gdoc'

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

zip_path = Path("stage_3B_surrogate_training_outputs.zip")

output_files = [
    "models/stage_3B_tmax_random_forest_model.pkl",
    "reports/stage_3B_surrogate_training_report.txt",
    "dataset/comsol/processed/stage_3B_train_test_predictions.csv",
    "figures/stage_3B_surrogate_training/predicted_vs_actual_Tmax.png",
    "figures/stage_3B_surrogate_training/error_distribution_Tmax.png",
    "figures/stage_3B_surrogate_training/feature_importance_Tmax.png",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for file in output_files:
        path = Path(file)
        if path.exists():
            z.write(path, arcname=file)
            print("Added:", file)
        else:
            print("Missing:", file)

files.download(str(zip_path))

Added: models/stage_3B_tmax_random_forest_model.pkl
Added: reports/stage_3B_surrogate_training_report.txt
Added: dataset/comsol/processed/stage_3B_train_test_predictions.csv
Added: figures/stage_3B_surrogate_training/predicted_vs_actual_Tmax.png
Added: figures/stage_3B_surrogate_training/error_distribution_Tmax.png
Added: figures/stage_3B_surrogate_training/feature_importance_Tmax.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path

dataset_path = Path("dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv")
print("FOUND" if dataset_path.exists() else "MISSING")
print(dataset_path)

MISSING
dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import shutil

source_file = Path("/content/drive/MyDrive/stage_2C_cleaned_for_surrogate.csv")

target_folder = Path("dataset/comsol/processed")
target_folder.mkdir(parents=True, exist_ok=True)

target_file = target_folder / "stage_2C_cleaned_for_surrogate.csv"

shutil.copy(source_file, target_file)

print("Copied:", target_file)
print("FOUND" if target_file.exists() else "MISSING")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/stage_2C_cleaned_for_surrogate.csv'

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

for file in drive_root.glob("stage_*.csv"):
    print(file.name)

stage_2C_operating_condition_sweep_temperature_summary_time_results.csv


In [3]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

for file in drive_root.glob("stage_*.csv"):
    print(file.name)

stage_2C_operating_condition_sweep_temperature_summary_time_results.csv
stage_2C_cleaned_for_surrogate.csv


In [4]:
from pathlib import Path
import shutil

source_file = Path("/content/drive/MyDrive/stage_2C_cleaned_for_surrogate.csv")
target_file = Path("dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv")

target_file.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(source_file, target_file)

print("FOUND" if target_file.exists() else "MISSING")
print(target_file)

FOUND
dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv


In [5]:
"""
Stage 3C Compact Neural Network Surrogate Training Script
Project: FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction
Author: Frank Ouma
Contact: +254725582132

Purpose:
This script trains a compact neural network surrogate model for predicting maximum
battery cell temperature from COMSOL-derived operating-condition data.

This stage comes after Stage 3B Random Forest training. The Random Forest model
proved that the COMSOL dataset is learnable. Stage 3C now trains a smaller,
more hardware-oriented model that can later be prepared for quantization and
FPGA implementation.

Input dataset:
    dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv

Inputs:
    I_pack_A
    h_conv_W_per_m2K
    Tamb_C
    time_s

Target:
    Tmax_C

Generated outputs:
    models/stage_3C_tmax_compact_neural_network.keras
    models/stage_3C_tmax_input_scaler.pkl
    models/stage_3C_tmax_output_scaler.pkl
    dataset/comsol/processed/stage_3C_neural_network_predictions.csv
    reports/stage_3C_compact_neural_network_training_report.txt
    figures/stage_3C_compact_neural_network/training_loss_curve.png
    figures/stage_3C_compact_neural_network/predicted_vs_actual_Tmax.png
    figures/stage_3C_compact_neural_network/error_distribution_Tmax.png
"""

from pathlib import Path
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


PROJECT_NAME = "FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction"
AUTHOR = "Frank Ouma"
CONTACT = "+254725582132"

FEATURE_COLUMNS = [
    "I_pack_A",
    "h_conv_W_per_m2K",
    "Tamb_C",
    "time_s",
]

TARGET_COLUMN = "Tmax_C"


def find_project_root() -> Path:
    current = Path.cwd().resolve()

    if (current / "dataset").exists():
        return current

    for parent in current.parents:
        if (parent / "dataset").exists():
            return parent

    return current


def set_reproducibility(seed: int = 42) -> None:
    np.random.seed(seed)
    tf.random.set_seed(seed)


def load_dataset(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {csv_path}\n"
            "Expected file: dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv"
        )

    df = pd.read_csv(csv_path)
    return df


def validate_dataset(df: pd.DataFrame) -> pd.DataFrame:
    required = FEATURE_COLUMNS + [TARGET_COLUMN]
    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError("Missing required columns: " + ", ".join(missing))

    clean_df = df.copy()
    for col in required:
        clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

    if clean_df[required].isna().sum().sum() > 0:
        raise ValueError(
            "The dataset contains missing or non-numeric values in required columns:\n"
            + clean_df[required].isna().sum().to_string()
        )

    return clean_df


def prepare_data(df: pd.DataFrame):
    X = df[FEATURE_COLUMNS].values.astype(np.float32)
    y = df[[TARGET_COLUMN]].values.astype(np.float32)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        shuffle=True,
    )

    input_scaler = StandardScaler()
    output_scaler = StandardScaler()

    X_train_scaled = input_scaler.fit_transform(X_train).astype(np.float32)
    X_test_scaled = input_scaler.transform(X_test).astype(np.float32)

    y_train_scaled = output_scaler.fit_transform(y_train).astype(np.float32)
    y_test_scaled = output_scaler.transform(y_test).astype(np.float32)

    return (
        X_train,
        X_test,
        y_train,
        y_test,
        X_train_scaled,
        X_test_scaled,
        y_train_scaled,
        y_test_scaled,
        input_scaler,
        output_scaler,
    )


def build_compact_neural_network(input_dim: int) -> keras.Model:
    model = keras.Sequential(
        [
            layers.Input(shape=(input_dim,), name="input_operating_conditions"),
            layers.Dense(16, activation="relu", name="hidden_layer_1"),
            layers.Dense(8, activation="relu", name="hidden_layer_2"),
            layers.Dense(1, activation="linear", name="predicted_Tmax_scaled"),
        ]
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"],
    )

    return model


def train_model(model, X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled):
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=40,
        restore_best_weights=True,
    )

    history = model.fit(
        X_train_scaled,
        y_train_scaled,
        validation_data=(X_test_scaled, y_test_scaled),
        epochs=600,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1,
    )

    return history


def evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test, output_scaler):
    train_pred_scaled = model.predict(X_train_scaled, verbose=0)
    test_pred_scaled = model.predict(X_test_scaled, verbose=0)

    train_pred = output_scaler.inverse_transform(train_pred_scaled)
    test_pred = output_scaler.inverse_transform(test_pred_scaled)

    train_rmse = mean_squared_error(y_train, train_pred) ** 0.5
    test_rmse = mean_squared_error(y_test, test_pred) ** 0.5

    metrics = {
        "train_mae_C": mean_absolute_error(y_train, train_pred),
        "test_mae_C": mean_absolute_error(y_test, test_pred),
        "train_rmse_C": train_rmse,
        "test_rmse_C": test_rmse,
        "train_r2": r2_score(y_train, train_pred),
        "test_r2": r2_score(y_test, test_pred),
        "train_predictions": train_pred.reshape(-1),
        "test_predictions": test_pred.reshape(-1),
    }

    return metrics


def create_prediction_table(X_train, X_test, y_train, y_test, metrics) -> pd.DataFrame:
    train_df = pd.DataFrame(X_train, columns=FEATURE_COLUMNS)
    train_df["dataset_split"] = "train"
    train_df["actual_Tmax_C"] = y_train.reshape(-1)
    train_df["predicted_Tmax_C"] = metrics["train_predictions"]

    test_df = pd.DataFrame(X_test, columns=FEATURE_COLUMNS)
    test_df["dataset_split"] = "test"
    test_df["actual_Tmax_C"] = y_test.reshape(-1)
    test_df["predicted_Tmax_C"] = metrics["test_predictions"]

    prediction_df = pd.concat([train_df, test_df], ignore_index=True)
    prediction_df["prediction_error_C"] = prediction_df["predicted_Tmax_C"] - prediction_df["actual_Tmax_C"]
    prediction_df["absolute_error_C"] = prediction_df["prediction_error_C"].abs()

    return prediction_df


def save_scaler(scaler, path: Path) -> None:
    with path.open("wb") as f:
        pickle.dump(scaler, f)


def save_report(report_path: Path, df: pd.DataFrame, model: keras.Model, metrics: dict, history) -> None:
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    final_train_loss = float(history.history["loss"][-1])
    final_val_loss = float(history.history["val_loss"][-1])

    with report_path.open("w", encoding="utf-8") as f:
        f.write("Stage 3C Compact Neural Network Surrogate Training Report\n")
        f.write(f"Project: {PROJECT_NAME}\n")
        f.write(f"Author: {AUTHOR}\n")
        f.write(f"Contact: {CONTACT}\n")
        f.write("\n")

        f.write("Purpose\n")
        f.write("This stage trains a compact neural network surrogate model for predicting COMSOL-derived maximum battery temperature.\n")
        f.write("The model is intended as the first bridge toward quantization and FPGA-oriented inference.\n")
        f.write("\n")

        f.write("Dataset\n")
        f.write(f"Rows used: {len(df)}\n")
        f.write(f"Input features: {FEATURE_COLUMNS}\n")
        f.write(f"Prediction target: {TARGET_COLUMN}\n")
        f.write("\n")

        f.write("Model Architecture\n")
        f.write("Input dimension: 4\n")
        f.write("Hidden layer 1: Dense, 16 neurons, ReLU activation\n")
        f.write("Hidden layer 2: Dense, 8 neurons, ReLU activation\n")
        f.write("Output layer: Dense, 1 neuron, linear activation\n")
        f.write(f"Total trainable parameters: {model.count_params()}\n")
        f.write("\n")

        f.write("Training Details\n")
        f.write("Optimizer: Adam\n")
        f.write("Learning rate: 0.001\n")
        f.write("Loss function: Mean Squared Error\n")
        f.write("Batch size: 32\n")
        f.write("Maximum epochs: 600\n")
        f.write("Early stopping patience: 40 epochs\n")
        f.write(f"Best validation epoch: {best_epoch}\n")
        f.write(f"Final scaled train loss: {final_train_loss:.8f}\n")
        f.write(f"Final scaled validation loss: {final_val_loss:.8f}\n")
        f.write("\n")

        f.write("Performance Metrics\n")
        f.write(f"Train MAE: {metrics['train_mae_C']:.6f} degC\n")
        f.write(f"Test MAE: {metrics['test_mae_C']:.6f} degC\n")
        f.write(f"Train RMSE: {metrics['train_rmse_C']:.6f} degC\n")
        f.write(f"Test RMSE: {metrics['test_rmse_C']:.6f} degC\n")
        f.write(f"Train R2: {metrics['train_r2']:.6f}\n")
        f.write(f"Test R2: {metrics['test_r2']:.6f}\n")
        f.write("\n")

        f.write("Engineering Interpretation\n")
        f.write("This compact neural network uses a small number of trainable parameters and primarily relies on multiply-add operations.\n")
        f.write("That makes it more suitable for later fixed-point quantization than the Random Forest baseline.\n")
        f.write("The Random Forest remains the accuracy benchmark, while this neural network is the hardware-oriented surrogate candidate.\n")
        f.write("\n")

        f.write("Next Step\n")
        f.write("The next stage should export the trained neural network weights, biases, and scaling factors.\n")
        f.write("After that, the model can be quantized to int16 or int8 for hardware-oriented inference testing.\n")


def plot_training_loss(history, figure_path: Path) -> None:
    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], label="Training loss")
    plt.plot(history.history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Scaled MSE Loss")
    plt.title("Stage 3C Compact Neural Network Training Loss")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def plot_predicted_vs_actual(prediction_df: pd.DataFrame, figure_path: Path) -> None:
    test_df = prediction_df[prediction_df["dataset_split"] == "test"]

    plt.figure(figsize=(7, 7))
    plt.scatter(test_df["actual_Tmax_C"], test_df["predicted_Tmax_C"], alpha=0.75)

    min_val = min(test_df["actual_Tmax_C"].min(), test_df["predicted_Tmax_C"].min())
    max_val = max(test_df["actual_Tmax_C"].max(), test_df["predicted_Tmax_C"].max())
    plt.plot([min_val, max_val], [min_val, max_val], linewidth=2)

    plt.xlabel("Actual Tmax (degC)")
    plt.ylabel("Predicted Tmax (degC)")
    plt.title("Stage 3C Neural Network: Predicted vs Actual Tmax")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def plot_error_distribution(prediction_df: pd.DataFrame, figure_path: Path) -> None:
    test_df = prediction_df[prediction_df["dataset_split"] == "test"]

    plt.figure(figsize=(8, 5))
    plt.hist(test_df["prediction_error_C"], bins=30, edgecolor="black")
    plt.xlabel("Prediction Error (degC)")
    plt.ylabel("Count")
    plt.title("Stage 3C Neural Network: Test Error Distribution")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def run_example_prediction(model, input_scaler, output_scaler) -> float:
    example = pd.DataFrame(
        [
            {
                "I_pack_A": 40.0,
                "h_conv_W_per_m2K": 20.0,
                "Tamb_C": 30.0,
                "time_s": 600.0,
            }
        ]
    )

    X_example = example[FEATURE_COLUMNS].values.astype(np.float32)
    X_example_scaled = input_scaler.transform(X_example).astype(np.float32)
    pred_scaled = model.predict(X_example_scaled, verbose=0)
    pred = output_scaler.inverse_transform(pred_scaled)
    return float(pred[0, 0])


def main() -> None:
    set_reproducibility(42)

    project_root = find_project_root()

    dataset_path = project_root / "dataset" / "comsol" / "processed" / "stage_2C_cleaned_for_surrogate.csv"
    output_prediction_path = project_root / "dataset" / "comsol" / "processed" / "stage_3C_neural_network_predictions.csv"

    model_dir = project_root / "models"
    report_dir = project_root / "reports"
    figure_dir = project_root / "figures" / "stage_3C_compact_neural_network"

    model_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    output_prediction_path.parent.mkdir(parents=True, exist_ok=True)

    model_path = model_dir / "stage_3C_tmax_compact_neural_network.keras"
    input_scaler_path = model_dir / "stage_3C_tmax_input_scaler.pkl"
    output_scaler_path = model_dir / "stage_3C_tmax_output_scaler.pkl"
    report_path = report_dir / "stage_3C_compact_neural_network_training_report.txt"

    print("Stage 3C Compact Neural Network Surrogate Training")
    print(f"Project root: {project_root}")
    print(f"Reading dataset: {dataset_path}")

    df = load_dataset(dataset_path)
    df = validate_dataset(df)

    (
        X_train,
        X_test,
        y_train,
        y_test,
        X_train_scaled,
        X_test_scaled,
        y_train_scaled,
        y_test_scaled,
        input_scaler,
        output_scaler,
    ) = prepare_data(df)

    model = build_compact_neural_network(input_dim=len(FEATURE_COLUMNS))
    history = train_model(model, X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled)
    metrics = evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test, output_scaler)

    prediction_df = create_prediction_table(X_train, X_test, y_train, y_test, metrics)
    prediction_df.to_csv(output_prediction_path, index=False)

    model.save(model_path)
    save_scaler(input_scaler, input_scaler_path)
    save_scaler(output_scaler, output_scaler_path)

    save_report(report_path, df, model, metrics, history)

    plot_training_loss(history, figure_dir / "training_loss_curve.png")
    plot_predicted_vs_actual(prediction_df, figure_dir / "predicted_vs_actual_Tmax.png")
    plot_error_distribution(prediction_df, figure_dir / "error_distribution_Tmax.png")

    example_prediction = run_example_prediction(model, input_scaler, output_scaler)

    print("Training completed successfully.")
    print(f"Train MAE: {metrics['train_mae_C']:.6f} degC")
    print(f"Test MAE: {metrics['test_mae_C']:.6f} degC")
    print(f"Train RMSE: {metrics['train_rmse_C']:.6f} degC")
    print(f"Test RMSE: {metrics['test_rmse_C']:.6f} degC")
    print(f"Train R2: {metrics['train_r2']:.6f}")
    print(f"Test R2: {metrics['test_r2']:.6f}")
    print(f"Example prediction, I=40A, h=20W/m2K, Tamb=30C, t=600s: Tmax={example_prediction:.4f} degC")
    print(f"Saved neural network model: {model_path}")
    print(f"Saved input scaler: {input_scaler_path}")
    print(f"Saved output scaler: {output_scaler_path}")
    print(f"Saved report: {report_path}")
    print(f"Saved predictions: {output_prediction_path}")
    print(f"Saved figures: {figure_dir}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print("Execution failed.")
        print(str(exc))
        sys.exit(1)


Stage 3C Compact Neural Network Surrogate Training
Project root: /content
Reading dataset: /content/dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv
Epoch 1/600
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.1917 - mae: 0.8258 - val_loss: 0.8291 - val_mae: 0.6512
Epoch 2/600
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.7332 - mae: 0.5771 - val_loss: 0.5212 - val_mae: 0.4633
Epoch 3/600
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4682 - mae: 0.4242 - val_loss: 0.3363 - val_mae: 0.3782
Epoch 4/600
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.2960 - mae: 0.3400 - val_loss: 0.2051 - val_mae: 0.2972
Epoch 5/600
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1785 - mae: 0.2688 - val_loss: 0.1202 - val_mae: 0.2299
Epoch 6/600
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1098 - mae: 0.2138 - val_loss: 0.0786 - val_mae: 0.1903
Epoch 7/600
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0762 - mae: 0.1810 - val_loss: 0.0587 - val_mae: 0.1664
Epoch 8/600
55/55 ━━━

In [6]:
from pathlib import Path
import zipfile
from google.colab import files

zip_path = Path("stage_3C_compact_neural_network_outputs.zip")

output_files = [
    "models/stage_3C_tmax_compact_neural_network.keras",
    "models/stage_3C_tmax_input_scaler.pkl",
    "models/stage_3C_tmax_output_scaler.pkl",
    "reports/stage_3C_compact_neural_network_training_report.txt",
    "dataset/comsol/processed/stage_3C_neural_network_predictions.csv",
    "figures/stage_3C_compact_neural_network/training_loss_curve.png",
    "figures/stage_3C_compact_neural_network/predicted_vs_actual_Tmax.png",
    "figures/stage_3C_compact_neural_network/error_distribution_Tmax.png",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for file in output_files:
        path = Path(file)
        if path.exists():
            z.write(path, arcname=file)
            print("Added:", file)
        else:
            print("Missing:", file)

files.download(str(zip_path))

Added: models/stage_3C_tmax_compact_neural_network.keras
Added: models/stage_3C_tmax_input_scaler.pkl
Added: models/stage_3C_tmax_output_scaler.pkl
Added: reports/stage_3C_compact_neural_network_training_report.txt
Added: dataset/comsol/processed/stage_3C_neural_network_predictions.csv
Added: figures/stage_3C_compact_neural_network/training_loss_curve.png
Added: figures/stage_3C_compact_neural_network/predicted_vs_actual_Tmax.png
Added: figures/stage_3C_compact_neural_network/error_distribution_Tmax.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from pathlib import Path

files_to_check = [
    "models/stage_3C_tmax_compact_neural_network.keras",
    "models/stage_3C_tmax_input_scaler.pkl",
    "models/stage_3C_tmax_output_scaler.pkl",
]

for file in files_to_check:
    path = Path(file)
    print(file, "FOUND" if path.exists() else "MISSING")

models/stage_3C_tmax_compact_neural_network.keras FOUND
models/stage_3C_tmax_input_scaler.pkl FOUND
models/stage_3C_tmax_output_scaler.pkl FOUND


In [8]:
"""
Stage 3D Neural Network Parameter Export Script
Project: FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction
Author: Frank Ouma
Contact: +254725582132

Purpose:
This script exports the trained Stage 3C compact neural network internals:
weights, biases, scaler means, scaler standard deviations, and model structure.

This is the bridge between the TensorFlow model and later fixed-point / FPGA-oriented inference.

Required input files:
    models/stage_3C_tmax_compact_neural_network.keras
    models/stage_3C_tmax_input_scaler.pkl
    models/stage_3C_tmax_output_scaler.pkl

Generated outputs:
    models/stage_3D_exported_parameters/stage_3D_model_summary.txt
    models/stage_3D_exported_parameters/stage_3D_scaling_parameters.csv
    models/stage_3D_exported_parameters/stage_3D_input_scaler_mean.csv
    models/stage_3D_exported_parameters/stage_3D_input_scaler_std.csv
    models/stage_3D_exported_parameters/stage_3D_output_scaler_mean.csv
    models/stage_3D_exported_parameters/stage_3D_output_scaler_std.csv
    models/stage_3D_exported_parameters/stage_3D_hidden_layer_1_weights.csv
    models/stage_3D_exported_parameters/stage_3D_hidden_layer_1_biases.csv
    models/stage_3D_exported_parameters/stage_3D_hidden_layer_2_weights.csv
    models/stage_3D_exported_parameters/stage_3D_hidden_layer_2_biases.csv
    models/stage_3D_exported_parameters/stage_3D_output_layer_weights.csv
    models/stage_3D_exported_parameters/stage_3D_output_layer_biases.csv
    models/stage_3D_exported_parameters/stage_3D_all_parameters.npz
    reports/stage_3D_neural_network_parameter_export_report.txt
"""

from pathlib import Path
import sys
import pickle
import json

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras


PROJECT_NAME = "FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction"
AUTHOR = "Frank Ouma"
CONTACT = "+254725582132"

FEATURE_COLUMNS = [
    "I_pack_A",
    "h_conv_W_per_m2K",
    "Tamb_C",
    "time_s",
]

TARGET_COLUMN = "Tmax_C"


def find_project_root() -> Path:
    current = Path.cwd().resolve()

    if (current / "models").exists():
        return current

    for parent in current.parents:
        if (parent / "models").exists():
            return parent

    return current


def load_pickle(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

    with path.open("rb") as f:
        return pickle.load(f)


def load_model(model_path: Path) -> keras.Model:
    if not model_path.exists():
        raise FileNotFoundError(f"Missing required model file: {model_path}")

    model = keras.models.load_model(model_path)
    return model


def get_layer_parameters(model: keras.Model) -> dict:
    """Extract weights and biases from all Dense layers."""
    dense_layers = [layer for layer in model.layers if isinstance(layer, keras.layers.Dense)]

    if len(dense_layers) != 3:
        raise ValueError(
            f"Expected 3 Dense layers, but found {len(dense_layers)}. "
            "Check the Stage 3C model architecture."
        )

    exported = {}

    layer_names = [
        "hidden_layer_1",
        "hidden_layer_2",
        "output_layer",
    ]

    for export_name, layer in zip(layer_names, dense_layers):
        weights, biases = layer.get_weights()
        exported[f"{export_name}_weights"] = weights.astype(np.float32)
        exported[f"{export_name}_biases"] = biases.astype(np.float32)

    return exported


def save_array_csv(array: np.ndarray, path: Path, row_prefix: str = "row") -> None:
    """Save a numpy array as a CSV file."""
    array = np.asarray(array)

    if array.ndim == 1:
        df = pd.DataFrame({"index": np.arange(len(array)), "value": array})
    elif array.ndim == 2:
        df = pd.DataFrame(array)
        df.insert(0, "row", [f"{row_prefix}_{i}" for i in range(array.shape[0])])
    else:
        raise ValueError(f"Only 1D or 2D arrays can be exported to CSV. Got shape {array.shape}")

    df.to_csv(path, index=False)


def save_scaling_parameters(input_scaler, output_scaler, output_dir: Path) -> pd.DataFrame:
    """Save scaler parameters used for input normalization and output recovery."""
    input_mean = input_scaler.mean_.astype(np.float32)
    input_std = input_scaler.scale_.astype(np.float32)
    output_mean = output_scaler.mean_.astype(np.float32)
    output_std = output_scaler.scale_.astype(np.float32)

    input_df = pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "mean": input_mean,
            "std": input_std,
        }
    )

    output_df = pd.DataFrame(
        {
            "target": [TARGET_COLUMN],
            "mean": output_mean,
            "std": output_std,
        }
    )

    input_df.to_csv(output_dir / "stage_3D_input_scaler_parameters.csv", index=False)
    output_df.to_csv(output_dir / "stage_3D_output_scaler_parameters.csv", index=False)

    save_array_csv(input_mean, output_dir / "stage_3D_input_scaler_mean.csv")
    save_array_csv(input_std, output_dir / "stage_3D_input_scaler_std.csv")
    save_array_csv(output_mean, output_dir / "stage_3D_output_scaler_mean.csv")
    save_array_csv(output_std, output_dir / "stage_3D_output_scaler_std.csv")

    combined_rows = []
    for feature, mean, std in zip(FEATURE_COLUMNS, input_mean, input_std):
        combined_rows.append(
            {
                "type": "input",
                "name": feature,
                "mean": mean,
                "std": std,
            }
        )

    combined_rows.append(
        {
            "type": "output",
            "name": TARGET_COLUMN,
            "mean": float(output_mean[0]),
            "std": float(output_std[0]),
        }
    )

    combined_df = pd.DataFrame(combined_rows)
    combined_df.to_csv(output_dir / "stage_3D_scaling_parameters.csv", index=False)

    return combined_df


def save_model_summary(model: keras.Model, output_dir: Path) -> None:
    """Save model architecture and parameter count."""
    summary_lines = []
    model.summary(print_fn=lambda line: summary_lines.append(line))

    with (output_dir / "stage_3D_model_summary.txt").open("w", encoding="utf-8") as f:
        f.write("Stage 3D Model Summary\n")
        f.write(f"Project: {PROJECT_NAME}\n")
        f.write(f"Author: {AUTHOR}\n")
        f.write(f"Contact: {CONTACT}\n")
        f.write("\n")
        f.write("Keras Model Summary\n")
        f.write("\n".join(summary_lines))
        f.write("\n\n")
        f.write(f"Total trainable parameters: {model.count_params()}\n")


def save_architecture_json(model: keras.Model, output_dir: Path) -> None:
    """Save model architecture JSON."""
    architecture = json.loads(model.to_json())

    with (output_dir / "stage_3D_model_architecture.json").open("w", encoding="utf-8") as f:
        json.dump(architecture, f, indent=4)


def save_parameter_report(report_path: Path, model: keras.Model, layer_params: dict, scaling_df: pd.DataFrame) -> None:
    """Save professional export report."""
    with report_path.open("w", encoding="utf-8") as f:
        f.write("Stage 3D Neural Network Parameter Export Report\n")
        f.write(f"Project: {PROJECT_NAME}\n")
        f.write(f"Author: {AUTHOR}\n")
        f.write(f"Contact: {CONTACT}\n")
        f.write("\n")

        f.write("Purpose\n")
        f.write("This stage exports the trained Stage 3C neural network parameters for later quantization and FPGA-oriented inference.\n")
        f.write("\n")

        f.write("Model Structure\n")
        f.write("Input features: I_pack_A, h_conv_W_per_m2K, Tamb_C, time_s\n")
        f.write("Target output: Tmax_C\n")
        f.write("Architecture: 4 inputs, Dense 16 ReLU, Dense 8 ReLU, Dense 1 linear output\n")
        f.write(f"Total trainable parameters: {model.count_params()}\n")
        f.write("\n")

        f.write("Exported Layer Parameter Shapes\n")
        for name, array in layer_params.items():
            f.write(f"{name}: shape {array.shape}\n")
        f.write("\n")

        f.write("Scaling Parameters\n")
        f.write(scaling_df.to_string(index=False))
        f.write("\n\n")

        f.write("Inference Equation Sequence\n")
        f.write("1. Normalize each input: x_scaled = (x - input_mean) / input_std\n")
        f.write("2. Hidden layer 1: h1 = ReLU(x_scaled @ W1 + b1)\n")
        f.write("3. Hidden layer 2: h2 = ReLU(h1 @ W2 + b2)\n")
        f.write("4. Output layer scaled: y_scaled = h2 @ W3 + b3\n")
        f.write("5. Recover physical output: Tmax_C = y_scaled * output_std + output_mean\n")
        f.write("\n")

        f.write("Next Step\n")
        f.write("The next stage is Stage 3E, where these floating-point weights, biases, and scaling factors are converted into fixed-point int16 or int8 representations.\n")


def main() -> None:
    project_root = find_project_root()

    model_path = project_root / "models" / "stage_3C_tmax_compact_neural_network.keras"
    input_scaler_path = project_root / "models" / "stage_3C_tmax_input_scaler.pkl"
    output_scaler_path = project_root / "models" / "stage_3C_tmax_output_scaler.pkl"

    output_dir = project_root / "models" / "stage_3D_exported_parameters"
    report_dir = project_root / "reports"

    output_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)

    report_path = report_dir / "stage_3D_neural_network_parameter_export_report.txt"

    print("Stage 3D Neural Network Parameter Export")
    print(f"Project root: {project_root}")
    print(f"Loading model: {model_path}")

    model = load_model(model_path)
    input_scaler = load_pickle(input_scaler_path)
    output_scaler = load_pickle(output_scaler_path)

    layer_params = get_layer_parameters(model)

    save_model_summary(model, output_dir)
    save_architecture_json(model, output_dir)

    save_array_csv(layer_params["hidden_layer_1_weights"], output_dir / "stage_3D_hidden_layer_1_weights.csv", row_prefix="input")
    save_array_csv(layer_params["hidden_layer_1_biases"], output_dir / "stage_3D_hidden_layer_1_biases.csv")

    save_array_csv(layer_params["hidden_layer_2_weights"], output_dir / "stage_3D_hidden_layer_2_weights.csv", row_prefix="hidden1")
    save_array_csv(layer_params["hidden_layer_2_biases"], output_dir / "stage_3D_hidden_layer_2_biases.csv")

    save_array_csv(layer_params["output_layer_weights"], output_dir / "stage_3D_output_layer_weights.csv", row_prefix="hidden2")
    save_array_csv(layer_params["output_layer_biases"], output_dir / "stage_3D_output_layer_biases.csv")

    scaling_df = save_scaling_parameters(input_scaler, output_scaler, output_dir)

    np.savez(
        output_dir / "stage_3D_all_parameters.npz",
        hidden_layer_1_weights=layer_params["hidden_layer_1_weights"],
        hidden_layer_1_biases=layer_params["hidden_layer_1_biases"],
        hidden_layer_2_weights=layer_params["hidden_layer_2_weights"],
        hidden_layer_2_biases=layer_params["hidden_layer_2_biases"],
        output_layer_weights=layer_params["output_layer_weights"],
        output_layer_biases=layer_params["output_layer_biases"],
        input_mean=input_scaler.mean_.astype(np.float32),
        input_std=input_scaler.scale_.astype(np.float32),
        output_mean=output_scaler.mean_.astype(np.float32),
        output_std=output_scaler.scale_.astype(np.float32),
    )

    save_parameter_report(report_path, model, layer_params, scaling_df)

    print("Export completed successfully.")
    print(f"Export folder: {output_dir}")
    print(f"Report: {report_path}")
    print("Exported files:")
    for path in sorted(output_dir.iterdir()):
        print(" -", path)


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print("Execution failed.")
        print(str(exc))
        sys.exit(1)


Stage 3D Neural Network Parameter Export
Project root: /content
Loading model: /content/models/stage_3C_tmax_compact_neural_network.keras


Export completed successfully.
Export folder: /content/models/stage_3D_exported_parameters
Report: /content/reports/stage_3D_neural_network_parameter_export_report.txt
Exported files:
 - /content/models/stage_3D_exported_parameters/stage_3D_all_parameters.npz
 - /content/models/stage_3D_exported_parameters/stage_3D_hidden_layer_1_biases.csv
 - /content/models/stage_3D_exported_parameters/stage_3D_hidden_layer_1_weights.csv
 - /content/models/stage_3D_exported_parameters/stage_3D_hidden_layer_2_biases.csv
 - /content/models/stage_3D_exported_parameters/stage_3D_hidden_layer_2_weights.csv
 - /content/models/stage_3D_exported_parameters/stage_3D_input_scaler_mean.csv
 - /content/models/stage_3D_exported_parameters/stage_3D_input_scaler_parameters.csv
 - /content/models/stage_3D_exported_parameters/stage_3D_input_scaler_std.csv
 - /content/models/stage_3D_exported_parameters/stage_3D_model_architecture.json
 - /content/models/stage_3D_exported_parameters/stage_3D_model_summary.txt
 - /cont

In [9]:
from pathlib import Path
import zipfile
from google.colab import files

zip_path = Path("stage_3D_neural_network_parameter_exports.zip")

output_files = [
    "models/stage_3D_exported_parameters/stage_3D_model_summary.txt",
    "models/stage_3D_exported_parameters/stage_3D_model_architecture.json",
    "models/stage_3D_exported_parameters/stage_3D_scaling_parameters.csv",
    "models/stage_3D_exported_parameters/stage_3D_input_scaler_parameters.csv",
    "models/stage_3D_exported_parameters/stage_3D_output_scaler_parameters.csv",
    "models/stage_3D_exported_parameters/stage_3D_hidden_layer_1_weights.csv",
    "models/stage_3D_exported_parameters/stage_3D_hidden_layer_1_biases.csv",
    "models/stage_3D_exported_parameters/stage_3D_hidden_layer_2_weights.csv",
    "models/stage_3D_exported_parameters/stage_3D_hidden_layer_2_biases.csv",
    "models/stage_3D_exported_parameters/stage_3D_output_layer_weights.csv",
    "models/stage_3D_exported_parameters/stage_3D_output_layer_biases.csv",
    "models/stage_3D_exported_parameters/stage_3D_all_parameters.npz",
    "reports/stage_3D_neural_network_parameter_export_report.txt",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for file in output_files:
        path = Path(file)
        if path.exists():
            z.write(path, arcname=file)
            print("Added:", file)
        else:
            print("Missing:", file)

files.download(str(zip_path))

Added: models/stage_3D_exported_parameters/stage_3D_model_summary.txt
Added: models/stage_3D_exported_parameters/stage_3D_model_architecture.json
Added: models/stage_3D_exported_parameters/stage_3D_scaling_parameters.csv
Added: models/stage_3D_exported_parameters/stage_3D_input_scaler_parameters.csv
Added: models/stage_3D_exported_parameters/stage_3D_output_scaler_parameters.csv
Added: models/stage_3D_exported_parameters/stage_3D_hidden_layer_1_weights.csv
Added: models/stage_3D_exported_parameters/stage_3D_hidden_layer_1_biases.csv
Added: models/stage_3D_exported_parameters/stage_3D_hidden_layer_2_weights.csv
Added: models/stage_3D_exported_parameters/stage_3D_hidden_layer_2_biases.csv
Added: models/stage_3D_exported_parameters/stage_3D_output_layer_weights.csv
Added: models/stage_3D_exported_parameters/stage_3D_output_layer_biases.csv
Added: models/stage_3D_exported_parameters/stage_3D_all_parameters.npz
Added: reports/stage_3D_neural_network_parameter_export_report.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
"""
Stage 3E Int16 Quantization Test Script
Project: FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction
Author: Frank Ouma
Contact: +254725582132

Purpose:
This script takes the trained Stage 3C compact neural network and performs the
first hardware-oriented quantization test.

This stage answers the question:
    If we replace floating-point neural network weights and biases with int16
    representations, how much accuracy do we lose?

Important:
This is a quantization simulation stage. It quantizes weights and biases to int16,
then dequantizes them back during Python inference to measure the numerical error.
The next stage will move closer to full fixed-point integer-only inference.

Required input files:
    models/stage_3C_tmax_compact_neural_network.keras
    models/stage_3C_tmax_input_scaler.pkl
    models/stage_3C_tmax_output_scaler.pkl
    dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv

Generated outputs:
    models/stage_3E_int16_quantized_parameters/
        stage_3E_quantized_hidden_layer_1_weights_int16.csv
        stage_3E_quantized_hidden_layer_1_biases_int16.csv
        stage_3E_quantized_hidden_layer_2_weights_int16.csv
        stage_3E_quantized_hidden_layer_2_biases_int16.csv
        stage_3E_quantized_output_layer_weights_int16.csv
        stage_3E_quantized_output_layer_biases_int16.csv
        stage_3E_quantization_scales.csv
        stage_3E_all_quantized_parameters.npz

    dataset/comsol/processed/stage_3E_int16_quantization_predictions.csv
    reports/stage_3E_int16_quantization_report.txt
    figures/stage_3E_int16_quantization/float_vs_int16_predictions.png
    figures/stage_3E_int16_quantization/int16_quantization_error_distribution.png
"""

from pathlib import Path
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


PROJECT_NAME = "FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction"
AUTHOR = "Frank Ouma"
CONTACT = "+254725582132"

FEATURE_COLUMNS = [
    "I_pack_A",
    "h_conv_W_per_m2K",
    "Tamb_C",
    "time_s",
]

TARGET_COLUMN = "Tmax_C"
INT16_MAX = 32767
INT16_MIN = -32768


def find_project_root() -> Path:
    current = Path.cwd().resolve()

    if (current / "models").exists() and (current / "dataset").exists():
        return current

    for parent in current.parents:
        if (parent / "models").exists() and (parent / "dataset").exists():
            return parent

    return current


def load_pickle(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    with path.open("rb") as f:
        return pickle.load(f)


def load_model(model_path: Path) -> keras.Model:
    if not model_path.exists():
        raise FileNotFoundError(f"Missing required model file: {model_path}")
    return keras.models.load_model(model_path)


def load_dataset(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing required dataset: {csv_path}")

    df = pd.read_csv(csv_path)

    required = FEATURE_COLUMNS + [TARGET_COLUMN]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError("Dataset missing required columns: " + ", ".join(missing))

    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=required).reset_index(drop=True)
    return df


def extract_float_parameters(model: keras.Model) -> dict:
    dense_layers = [layer for layer in model.layers if isinstance(layer, keras.layers.Dense)]

    if len(dense_layers) != 3:
        raise ValueError(f"Expected 3 Dense layers but found {len(dense_layers)}")

    names = ["hidden_layer_1", "hidden_layer_2", "output_layer"]
    params = {}

    for name, layer in zip(names, dense_layers):
        weights, biases = layer.get_weights()
        params[f"{name}_weights"] = weights.astype(np.float32)
        params[f"{name}_biases"] = biases.astype(np.float32)

    return params


def quantize_array_symmetric_int16(array: np.ndarray):
    """
    Symmetric int16 quantization.

    Real value is approximated as:
        real_value ~= int16_value * scale

    scale is chosen so the largest absolute value maps close to 32767.
    """
    array = np.asarray(array, dtype=np.float32)
    max_abs = float(np.max(np.abs(array)))

    if max_abs == 0.0:
        scale = 1.0
        q = np.zeros_like(array, dtype=np.int16)
    else:
        scale = max_abs / INT16_MAX
        q = np.round(array / scale)
        q = np.clip(q, INT16_MIN, INT16_MAX).astype(np.int16)

    dequantized = q.astype(np.float32) * np.float32(scale)
    quantization_error = dequantized - array

    return q, scale, dequantized, quantization_error


def quantize_model_parameters(float_params: dict):
    quantized = {}
    dequantized = {}
    scales = []
    errors = {}

    for name, array in float_params.items():
        q, scale, dq, err = quantize_array_symmetric_int16(array)
        quantized[name] = q
        dequantized[name] = dq
        errors[name] = err
        scales.append(
            {
                "parameter": name,
                "shape": str(array.shape),
                "int_type": "int16",
                "scale": scale,
                "max_abs_float": float(np.max(np.abs(array))),
                "max_abs_quantization_error": float(np.max(np.abs(err))),
                "mean_abs_quantization_error": float(np.mean(np.abs(err))),
            }
        )

    scales_df = pd.DataFrame(scales)
    return quantized, dequantized, scales_df, errors


def relu(x: np.ndarray) -> np.ndarray:
    return np.maximum(x, 0.0)


def run_manual_inference(X_scaled: np.ndarray, params: dict) -> np.ndarray:
    """
    Manual neural network inference using supplied weights and biases.

    Architecture:
        h1 = ReLU(X @ W1 + b1)
        h2 = ReLU(h1 @ W2 + b2)
        y  = h2 @ W3 + b3
    """
    W1 = params["hidden_layer_1_weights"]
    b1 = params["hidden_layer_1_biases"]
    W2 = params["hidden_layer_2_weights"]
    b2 = params["hidden_layer_2_biases"]
    W3 = params["output_layer_weights"]
    b3 = params["output_layer_biases"]

    h1 = relu(np.matmul(X_scaled, W1) + b1)
    h2 = relu(np.matmul(h1, W2) + b2)
    y_scaled = np.matmul(h2, W3) + b3
    return y_scaled


def save_quantized_arrays(quantized: dict, output_dir: Path) -> None:
    for name, array in quantized.items():
        flat = np.asarray(array)

        if flat.ndim == 1:
            df = pd.DataFrame({"index": np.arange(flat.shape[0]), "int16_value": flat})
        elif flat.ndim == 2:
            df = pd.DataFrame(flat)
            df.insert(0, "row", np.arange(flat.shape[0]))
        else:
            raise ValueError(f"Unsupported array dimension for {name}: {flat.shape}")

        df.to_csv(output_dir / f"stage_3E_quantized_{name}_int16.csv", index=False)


def save_report(
    report_path: Path,
    scales_df: pd.DataFrame,
    metrics: dict,
    quantization_summary: dict,
) -> None:
    with report_path.open("w", encoding="utf-8") as f:
        f.write("Stage 3E Int16 Quantization Report\n")
        f.write(f"Project: {PROJECT_NAME}\n")
        f.write(f"Author: {AUTHOR}\n")
        f.write(f"Contact: {CONTACT}\n")
        f.write("\n")

        f.write("Purpose\n")
        f.write("This stage quantizes the compact neural network weights and biases to int16 and evaluates the resulting prediction difference.\n")
        f.write("\n")

        f.write("Quantization Method\n")
        f.write("Symmetric per-array int16 quantization was used.\n")
        f.write("Each floating-point array is converted using: int16_value = round(float_value / scale).\n")
        f.write("For verification, each int16 array is dequantized using: float_approx = int16_value * scale.\n")
        f.write("\n")

        f.write("Quantization Scales and Parameter Error\n")
        f.write(scales_df.to_string(index=False))
        f.write("\n\n")

        f.write("Float Model vs Int16-Quantized Model Prediction Difference\n")
        f.write(f"MAE between float and int16-dequantized predictions: {metrics['float_vs_int16_mae_C']:.8f} degC\n")
        f.write(f"RMSE between float and int16-dequantized predictions: {metrics['float_vs_int16_rmse_C']:.8f} degC\n")
        f.write(f"Maximum absolute prediction difference: {metrics['float_vs_int16_max_abs_error_C']:.8f} degC\n")
        f.write("\n")

        f.write("Int16-Quantized Model vs COMSOL Target\n")
        f.write(f"MAE vs COMSOL Tmax_C: {metrics['int16_vs_actual_mae_C']:.8f} degC\n")
        f.write(f"RMSE vs COMSOL Tmax_C: {metrics['int16_vs_actual_rmse_C']:.8f} degC\n")
        f.write(f"R2 vs COMSOL Tmax_C: {metrics['int16_vs_actual_r2']:.8f}\n")
        f.write("\n")

        f.write("Quantization Summary\n")
        for key, value in quantization_summary.items():
            f.write(f"{key}: {value}\n")
        f.write("\n")

        f.write("Engineering Interpretation\n")
        f.write("If the float-to-int16 prediction difference is very small, the neural network is suitable for int16 fixed-point development.\n")
        f.write("This stage does not yet prove full FPGA bit-exact inference. It proves that the trained weights and biases can be represented in int16 with small numerical loss.\n")
        f.write("\n")

        f.write("Next Step\n")
        f.write("The next stage should implement a full fixed-point inference simulation that also handles input scaling, intermediate layer arithmetic, ReLU, and output recovery in integer form.\n")


def plot_float_vs_int16(prediction_df: pd.DataFrame, figure_path: Path) -> None:
    plt.figure(figsize=(7, 7))
    plt.scatter(prediction_df["float_model_Tmax_C"], prediction_df["int16_dequantized_Tmax_C"], alpha=0.65)

    min_val = min(prediction_df["float_model_Tmax_C"].min(), prediction_df["int16_dequantized_Tmax_C"].min())
    max_val = max(prediction_df["float_model_Tmax_C"].max(), prediction_df["int16_dequantized_Tmax_C"].max())
    plt.plot([min_val, max_val], [min_val, max_val], linewidth=2)

    plt.xlabel("Float Model Tmax Prediction (degC)")
    plt.ylabel("Int16-Dequantized Tmax Prediction (degC)")
    plt.title("Stage 3E Float Model vs Int16-Quantized Model")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def plot_quantization_error(prediction_df: pd.DataFrame, figure_path: Path) -> None:
    plt.figure(figsize=(8, 5))
    plt.hist(prediction_df["float_minus_int16_error_C"], bins=40, edgecolor="black")
    plt.xlabel("Float Prediction - Int16-Dequantized Prediction (degC)")
    plt.ylabel("Count")
    plt.title("Stage 3E Int16 Quantization Prediction Error")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def main() -> None:
    project_root = find_project_root()

    model_path = project_root / "models" / "stage_3C_tmax_compact_neural_network.keras"
    input_scaler_path = project_root / "models" / "stage_3C_tmax_input_scaler.pkl"
    output_scaler_path = project_root / "models" / "stage_3C_tmax_output_scaler.pkl"
    dataset_path = project_root / "dataset" / "comsol" / "processed" / "stage_2C_cleaned_for_surrogate.csv"

    output_dir = project_root / "models" / "stage_3E_int16_quantized_parameters"
    report_dir = project_root / "reports"
    figure_dir = project_root / "figures" / "stage_3E_int16_quantization"
    prediction_output_path = project_root / "dataset" / "comsol" / "processed" / "stage_3E_int16_quantization_predictions.csv"

    output_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    prediction_output_path.parent.mkdir(parents=True, exist_ok=True)

    report_path = report_dir / "stage_3E_int16_quantization_report.txt"

    print("Stage 3E Int16 Quantization Test")
    print(f"Project root: {project_root}")

    model = load_model(model_path)
    input_scaler = load_pickle(input_scaler_path)
    output_scaler = load_pickle(output_scaler_path)
    df = load_dataset(dataset_path)

    X_raw = df[FEATURE_COLUMNS].values.astype(np.float32)
    y_actual = df[TARGET_COLUMN].values.astype(np.float32).reshape(-1, 1)
    X_scaled = input_scaler.transform(X_raw).astype(np.float32)

    float_pred_scaled = model.predict(X_scaled, verbose=0)
    float_pred_C = output_scaler.inverse_transform(float_pred_scaled)

    float_params = extract_float_parameters(model)
    quantized_params, dequantized_params, scales_df, errors = quantize_model_parameters(float_params)

    int16_pred_scaled = run_manual_inference(X_scaled, dequantized_params)
    int16_pred_C = output_scaler.inverse_transform(int16_pred_scaled)

    float_vs_int16_error = float_pred_C.reshape(-1) - int16_pred_C.reshape(-1)

    prediction_df = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()
    prediction_df["float_model_Tmax_C"] = float_pred_C.reshape(-1)
    prediction_df["int16_dequantized_Tmax_C"] = int16_pred_C.reshape(-1)
    prediction_df["float_minus_int16_error_C"] = float_vs_int16_error
    prediction_df["abs_float_minus_int16_error_C"] = np.abs(float_vs_int16_error)
    prediction_df["int16_vs_actual_error_C"] = int16_pred_C.reshape(-1) - y_actual.reshape(-1)
    prediction_df["abs_int16_vs_actual_error_C"] = np.abs(prediction_df["int16_vs_actual_error_C"])
    prediction_df.to_csv(prediction_output_path, index=False)

    metrics = {
        "float_vs_int16_mae_C": mean_absolute_error(float_pred_C, int16_pred_C),
        "float_vs_int16_rmse_C": mean_squared_error(float_pred_C, int16_pred_C) ** 0.5,
        "float_vs_int16_max_abs_error_C": float(np.max(np.abs(float_vs_int16_error))),
        "int16_vs_actual_mae_C": mean_absolute_error(y_actual, int16_pred_C),
        "int16_vs_actual_rmse_C": mean_squared_error(y_actual, int16_pred_C) ** 0.5,
        "int16_vs_actual_r2": r2_score(y_actual, int16_pred_C),
    }

    save_quantized_arrays(quantized_params, output_dir)
    scales_df.to_csv(output_dir / "stage_3E_quantization_scales.csv", index=False)

    np.savez(
        output_dir / "stage_3E_all_quantized_parameters.npz",
        hidden_layer_1_weights_int16=quantized_params["hidden_layer_1_weights"],
        hidden_layer_1_biases_int16=quantized_params["hidden_layer_1_biases"],
        hidden_layer_2_weights_int16=quantized_params["hidden_layer_2_weights"],
        hidden_layer_2_biases_int16=quantized_params["hidden_layer_2_biases"],
        output_layer_weights_int16=quantized_params["output_layer_weights"],
        output_layer_biases_int16=quantized_params["output_layer_biases"],
        hidden_layer_1_weights_scale=scales_df.loc[scales_df["parameter"] == "hidden_layer_1_weights", "scale"].values[0],
        hidden_layer_1_biases_scale=scales_df.loc[scales_df["parameter"] == "hidden_layer_1_biases", "scale"].values[0],
        hidden_layer_2_weights_scale=scales_df.loc[scales_df["parameter"] == "hidden_layer_2_weights", "scale"].values[0],
        hidden_layer_2_biases_scale=scales_df.loc[scales_df["parameter"] == "hidden_layer_2_biases", "scale"].values[0],
        output_layer_weights_scale=scales_df.loc[scales_df["parameter"] == "output_layer_weights", "scale"].values[0],
        output_layer_biases_scale=scales_df.loc[scales_df["parameter"] == "output_layer_biases", "scale"].values[0],
    )

    quantization_summary = {
        "int_type": "int16",
        "quantization_style": "symmetric per-array quantization",
        "number_of_quantized_arrays": len(quantized_params),
        "prediction_rows_evaluated": len(prediction_df),
    }

    save_report(report_path, scales_df, metrics, quantization_summary)
    plot_float_vs_int16(prediction_df, figure_dir / "float_vs_int16_predictions.png")
    plot_quantization_error(prediction_df, figure_dir / "int16_quantization_error_distribution.png")

    print("Quantization test completed successfully.")
    print(f"Float vs int16 MAE: {metrics['float_vs_int16_mae_C']:.10f} degC")
    print(f"Float vs int16 RMSE: {metrics['float_vs_int16_rmse_C']:.10f} degC")
    print(f"Float vs int16 max absolute error: {metrics['float_vs_int16_max_abs_error_C']:.10f} degC")
    print(f"Int16 vs actual COMSOL MAE: {metrics['int16_vs_actual_mae_C']:.10f} degC")
    print(f"Int16 vs actual COMSOL RMSE: {metrics['int16_vs_actual_rmse_C']:.10f} degC")
    print(f"Int16 vs actual COMSOL R2: {metrics['int16_vs_actual_r2']:.10f}")
    print(f"Quantized parameter folder: {output_dir}")
    print(f"Prediction comparison file: {prediction_output_path}")
    print(f"Report: {report_path}")
    print(f"Figures: {figure_dir}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print("Execution failed.")
        print(str(exc))
        sys.exit(1)


Stage 3E Int16 Quantization Test
Project root: /content
Quantization test completed successfully.
Float vs int16 MAE: 0.0007309662 degC
Float vs int16 RMSE: 0.0009326446 degC
Float vs int16 max absolute error: 0.0030059814 degC
Int16 vs actual COMSOL MAE: 0.4502785802 degC
Int16 vs actual COMSOL RMSE: 0.5939009876 degC
Int16 vs actual COMSOL R2: 0.9994687438
Quantized parameter folder: /content/models/stage_3E_int16_quantized_parameters
Prediction comparison file: /content/dataset/comsol/processed/stage_3E_int16_quantization_predictions.csv
Report: /content/reports/stage_3E_int16_quantization_report.txt
Figures: /content/figures/stage_3E_int16_quantization


In [11]:
from pathlib import Path
import zipfile
from google.colab import files

zip_path = Path("stage_3E_int16_quantization_outputs.zip")

output_files = [
    "models/stage_3E_int16_quantized_parameters/stage_3E_quantized_hidden_layer_1_weights_int16.csv",
    "models/stage_3E_int16_quantized_parameters/stage_3E_quantized_hidden_layer_1_biases_int16.csv",
    "models/stage_3E_int16_quantized_parameters/stage_3E_quantized_hidden_layer_2_weights_int16.csv",
    "models/stage_3E_int16_quantized_parameters/stage_3E_quantized_hidden_layer_2_biases_int16.csv",
    "models/stage_3E_int16_quantized_parameters/stage_3E_quantized_output_layer_weights_int16.csv",
    "models/stage_3E_int16_quantized_parameters/stage_3E_quantized_output_layer_biases_int16.csv",
    "models/stage_3E_int16_quantized_parameters/stage_3E_quantization_scales.csv",
    "models/stage_3E_int16_quantized_parameters/stage_3E_all_quantized_parameters.npz",
    "dataset/comsol/processed/stage_3E_int16_quantization_predictions.csv",
    "reports/stage_3E_int16_quantization_report.txt",
    "figures/stage_3E_int16_quantization/float_vs_int16_predictions.png",
    "figures/stage_3E_int16_quantization/int16_quantization_error_distribution.png",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for file in output_files:
        path = Path(file)
        if path.exists():
            z.write(path, arcname=file)
            print("Added:", file)
        else:
            print("Missing:", file)

files.download(str(zip_path))

Added: models/stage_3E_int16_quantized_parameters/stage_3E_quantized_hidden_layer_1_weights_int16.csv
Added: models/stage_3E_int16_quantized_parameters/stage_3E_quantized_hidden_layer_1_biases_int16.csv
Added: models/stage_3E_int16_quantized_parameters/stage_3E_quantized_hidden_layer_2_weights_int16.csv
Added: models/stage_3E_int16_quantized_parameters/stage_3E_quantized_hidden_layer_2_biases_int16.csv
Added: models/stage_3E_int16_quantized_parameters/stage_3E_quantized_output_layer_weights_int16.csv
Added: models/stage_3E_int16_quantized_parameters/stage_3E_quantized_output_layer_biases_int16.csv
Added: models/stage_3E_int16_quantized_parameters/stage_3E_quantization_scales.csv
Added: models/stage_3E_int16_quantized_parameters/stage_3E_all_quantized_parameters.npz
Added: dataset/comsol/processed/stage_3E_int16_quantization_predictions.csv
Added: reports/stage_3E_int16_quantization_report.txt
Added: figures/stage_3E_int16_quantization/float_vs_int16_predictions.png
Added: figures/stage

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
"""
Stage 3F Fixed-Point Inference Simulation Script
Project: FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction
Author: Frank Ouma
Contact: +254725582132

Purpose:
This script moves beyond simple int16 weight quantization and simulates a more
hardware-realistic fixed-point neural network inference flow.

Stage 3E answered:
    Can the trained floating-point weights and biases survive int16 quantization?

Stage 3F answers:
    If the inputs, intermediate layer values, ReLU outputs, weights, biases,
    and output recovery are handled with fixed-point-style integer arithmetic,
    does the model still remain accurate enough?

This is a critical bridge toward FPGA implementation because an FPGA will not
naturally run Python floats. It will run fixed-width arithmetic such as int16,
int32 accumulators, shifts, scaling, saturation, and ReLU comparisons.

Required input files:
    models/stage_3C_tmax_compact_neural_network.keras
    models/stage_3C_tmax_input_scaler.pkl
    models/stage_3C_tmax_output_scaler.pkl
    dataset/comsol/processed/stage_2C_cleaned_for_surrogate.csv

Generated outputs:
    dataset/comsol/processed/stage_3F_fixed_point_inference_predictions.csv
    reports/stage_3F_fixed_point_inference_report.txt
    figures/stage_3F_fixed_point_inference/float_vs_fixed_point_predictions.png
    figures/stage_3F_fixed_point_inference/fixed_point_error_distribution.png
    models/stage_3F_fixed_point_parameters/stage_3F_fixed_point_configuration.txt
    models/stage_3F_fixed_point_parameters/stage_3F_fixed_point_scales.csv
"""

from pathlib import Path
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


PROJECT_NAME = "FPGA-Accelerated Battery Thermal Digital Twin for Real-Time Safety Prediction"
AUTHOR = "Frank Ouma"
CONTACT = "+254725582132"

FEATURE_COLUMNS = [
    "I_pack_A",
    "h_conv_W_per_m2K",
    "Tamb_C",
    "time_s",
]

TARGET_COLUMN = "Tmax_C"

# Fixed-point configuration.
# Q format explanation:
# A value stored as integer q represents real value q / SCALE.
# SCALE = 2^FRACTIONAL_BITS.
# Example with FRACTIONAL_BITS = 12:
# integer 4096 represents 1.0.
FRACTIONAL_BITS = 12
SCALE = 2 ** FRACTIONAL_BITS

INT16_MIN = -32768
INT16_MAX = 32767
INT32_MIN = -2147483648
INT32_MAX = 2147483647


def find_project_root() -> Path:
    current = Path.cwd().resolve()

    if (current / "models").exists() and (current / "dataset").exists():
        return current

    for parent in current.parents:
        if (parent / "models").exists() and (parent / "dataset").exists():
            return parent

    return current


def load_pickle(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    with path.open("rb") as f:
        return pickle.load(f)


def load_model(model_path: Path) -> keras.Model:
    if not model_path.exists():
        raise FileNotFoundError(f"Missing required model file: {model_path}")
    return keras.models.load_model(model_path)


def load_dataset(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing dataset: {csv_path}")

    df = pd.read_csv(csv_path)
    required = FEATURE_COLUMNS + [TARGET_COLUMN]

    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError("Dataset missing required columns: " + ", ".join(missing))

    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=required).reset_index(drop=True)
    return df


def extract_float_parameters(model: keras.Model) -> dict:
    dense_layers = [layer for layer in model.layers if isinstance(layer, keras.layers.Dense)]

    if len(dense_layers) != 3:
        raise ValueError(f"Expected 3 Dense layers but found {len(dense_layers)}")

    names = ["hidden_layer_1", "hidden_layer_2", "output_layer"]
    params = {}

    for name, layer in zip(names, dense_layers):
        weights, biases = layer.get_weights()
        params[f"{name}_weights"] = weights.astype(np.float32)
        params[f"{name}_biases"] = biases.astype(np.float32)

    return params


def to_fixed_q(value: np.ndarray, scale: int = SCALE, dtype=np.int16) -> np.ndarray:
    """Convert floating-point values to fixed-point integer representation."""
    q = np.round(value * scale)

    if dtype == np.int16:
        q = np.clip(q, INT16_MIN, INT16_MAX).astype(np.int16)
    elif dtype == np.int32:
        q = np.clip(q, INT32_MIN, INT32_MAX).astype(np.int32)
    else:
        q = q.astype(dtype)

    return q


def from_fixed_q(q_value: np.ndarray, scale: int = SCALE) -> np.ndarray:
    """Convert fixed-point integer representation back to floating-point for evaluation."""
    return q_value.astype(np.float32) / np.float32(scale)


def fixed_matmul_q(a_q: np.ndarray, w_q: np.ndarray, b_q: np.ndarray) -> np.ndarray:
    """
    Fixed-point dense layer simulation.

    a_q and w_q are Q-format int16 arrays.
    Multiplication produces Q^2 scale, so after accumulation we shift right by
    FRACTIONAL_BITS to return to Q-format.

    Equivalent real operation:
        y = a @ w + b

    Simulated fixed-point operation:
        acc_int32 = sum(a_q * w_q)
        y_q = (acc_int32 >> FRACTIONAL_BITS) + b_q
    """
    a_i32 = a_q.astype(np.int32)
    w_i32 = w_q.astype(np.int32)
    b_i32 = b_q.astype(np.int32)

    acc = np.matmul(a_i32, w_i32)

    # Integer rescaling back to Q format.
    y_q = (acc >> FRACTIONAL_BITS) + b_i32

    # Saturate to int16 range for layer output storage.
    y_q = np.clip(y_q, INT16_MIN, INT16_MAX).astype(np.int16)

    return y_q


def relu_q(x_q: np.ndarray) -> np.ndarray:
    """Fixed-point ReLU: negative integer values become zero."""
    return np.maximum(x_q, 0).astype(np.int16)


def run_fixed_point_inference_q(X_scaled_float: np.ndarray, float_params: dict) -> np.ndarray:
    """
    Full fixed-point-style inference simulation using Q-format integers.

    Inputs are already normalized using the trained StandardScaler.
    Then inputs, weights, and biases are converted into Q-format fixed-point.
    Each layer is computed using int32 accumulation and int16 saturated outputs.
    """
    X_q = to_fixed_q(X_scaled_float, dtype=np.int16)

    W1_q = to_fixed_q(float_params["hidden_layer_1_weights"], dtype=np.int16)
    b1_q = to_fixed_q(float_params["hidden_layer_1_biases"], dtype=np.int16)

    W2_q = to_fixed_q(float_params["hidden_layer_2_weights"], dtype=np.int16)
    b2_q = to_fixed_q(float_params["hidden_layer_2_biases"], dtype=np.int16)

    W3_q = to_fixed_q(float_params["output_layer_weights"], dtype=np.int16)
    b3_q = to_fixed_q(float_params["output_layer_biases"], dtype=np.int16)

    h1_q = fixed_matmul_q(X_q, W1_q, b1_q)
    h1_q = relu_q(h1_q)

    h2_q = fixed_matmul_q(h1_q, W2_q, b2_q)
    h2_q = relu_q(h2_q)

    y_scaled_q = fixed_matmul_q(h2_q, W3_q, b3_q)

    # Return scaled output as float for physical Tmax recovery through output scaler.
    y_scaled_float_approx = from_fixed_q(y_scaled_q)

    return y_scaled_float_approx


def save_fixed_point_configuration(output_dir: Path) -> pd.DataFrame:
    rows = [
        {"parameter": "fractional_bits", "value": FRACTIONAL_BITS},
        {"parameter": "scale", "value": SCALE},
        {"parameter": "input_storage", "value": "int16"},
        {"parameter": "weight_storage", "value": "int16"},
        {"parameter": "bias_storage", "value": "int16"},
        {"parameter": "accumulator", "value": "int32"},
        {"parameter": "activation", "value": "ReLU integer max(x,0)"},
        {"parameter": "layer_output_storage", "value": "int16 saturated"},
    ]

    df = pd.DataFrame(rows)
    df.to_csv(output_dir / "stage_3F_fixed_point_scales.csv", index=False)

    with (output_dir / "stage_3F_fixed_point_configuration.txt").open("w", encoding="utf-8") as f:
        f.write("Stage 3F Fixed-Point Configuration\n")
        f.write(f"Project: {PROJECT_NAME}\n")
        f.write(f"Author: {AUTHOR}\n")
        f.write(f"Contact: {CONTACT}\n")
        f.write("\n")
        f.write(f"Fixed-point format: Q format with {FRACTIONAL_BITS} fractional bits\n")
        f.write(f"Scale: {SCALE}\n")
        f.write("Real value approximation: real_value = integer_value / scale\n")
        f.write("Integer conversion: integer_value = round(real_value * scale)\n")
        f.write("Layer multiplication uses int32 accumulation.\n")
        f.write("Layer outputs are rescaled by right-shifting by the number of fractional bits.\n")
        f.write("ReLU is implemented as max(integer_value, 0).\n")

    return df


def save_report(report_path: Path, metrics: dict, config_df: pd.DataFrame) -> None:
    with report_path.open("w", encoding="utf-8") as f:
        f.write("Stage 3F Fixed-Point Inference Simulation Report\n")
        f.write(f"Project: {PROJECT_NAME}\n")
        f.write(f"Author: {AUTHOR}\n")
        f.write(f"Contact: {CONTACT}\n")
        f.write("\n")

        f.write("Purpose\n")
        f.write("This stage simulates FPGA-style fixed-point inference using int16 storage and int32 accumulation.\n")
        f.write("It evaluates whether the trained neural network remains accurate when arithmetic is constrained closer to hardware behavior.\n")
        f.write("\n")

        f.write("Fixed-Point Configuration\n")
        f.write(config_df.to_string(index=False))
        f.write("\n\n")

        f.write("Float Model vs Fixed-Point Simulation\n")
        f.write(f"MAE: {metrics['float_vs_fixed_mae_C']:.8f} degC\n")
        f.write(f"RMSE: {metrics['float_vs_fixed_rmse_C']:.8f} degC\n")
        f.write(f"Max absolute error: {metrics['float_vs_fixed_max_abs_error_C']:.8f} degC\n")
        f.write("\n")

        f.write("Fixed-Point Simulation vs COMSOL Target\n")
        f.write(f"MAE: {metrics['fixed_vs_actual_mae_C']:.8f} degC\n")
        f.write(f"RMSE: {metrics['fixed_vs_actual_rmse_C']:.8f} degC\n")
        f.write(f"R2: {metrics['fixed_vs_actual_r2']:.8f}\n")
        f.write("\n")

        f.write("Engineering Interpretation\n")
        f.write("This stage is more realistic than simple parameter quantization because it constrains layer computations using integer-like Q-format arithmetic.\n")
        f.write("If errors remain low, the neural network is a strong candidate for FPGA implementation.\n")
        f.write("If errors increase significantly, the fixed-point scaling strategy must be refined before hardware deployment.\n")
        f.write("\n")

        f.write("Next Step\n")
        f.write("After this stage, the next step is to export a hardware inference specification and begin mapping the fixed-point neural network to Verilog/SystemVerilog modules.\n")


def plot_float_vs_fixed(prediction_df: pd.DataFrame, figure_path: Path) -> None:
    plt.figure(figsize=(7, 7))
    plt.scatter(prediction_df["float_model_Tmax_C"], prediction_df["fixed_point_Tmax_C"], alpha=0.65)

    min_val = min(prediction_df["float_model_Tmax_C"].min(), prediction_df["fixed_point_Tmax_C"].min())
    max_val = max(prediction_df["float_model_Tmax_C"].max(), prediction_df["fixed_point_Tmax_C"].max())
    plt.plot([min_val, max_val], [min_val, max_val], linewidth=2)

    plt.xlabel("Float Model Tmax Prediction (degC)")
    plt.ylabel("Fixed-Point Tmax Prediction (degC)")
    plt.title("Stage 3F Float Model vs Fixed-Point Simulation")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def plot_error_distribution(prediction_df: pd.DataFrame, figure_path: Path) -> None:
    plt.figure(figsize=(8, 5))
    plt.hist(prediction_df["float_minus_fixed_error_C"], bins=40, edgecolor="black")
    plt.xlabel("Float Prediction - Fixed-Point Prediction (degC)")
    plt.ylabel("Count")
    plt.title("Stage 3F Fixed-Point Prediction Error Distribution")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.close()


def main() -> None:
    project_root = find_project_root()

    model_path = project_root / "models" / "stage_3C_tmax_compact_neural_network.keras"
    input_scaler_path = project_root / "models" / "stage_3C_tmax_input_scaler.pkl"
    output_scaler_path = project_root / "models" / "stage_3C_tmax_output_scaler.pkl"
    dataset_path = project_root / "dataset" / "comsol" / "processed" / "stage_2C_cleaned_for_surrogate.csv"

    parameter_dir = project_root / "models" / "stage_3F_fixed_point_parameters"
    report_dir = project_root / "reports"
    figure_dir = project_root / "figures" / "stage_3F_fixed_point_inference"
    prediction_output_path = project_root / "dataset" / "comsol" / "processed" / "stage_3F_fixed_point_inference_predictions.csv"

    parameter_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    prediction_output_path.parent.mkdir(parents=True, exist_ok=True)

    report_path = report_dir / "stage_3F_fixed_point_inference_report.txt"

    print("Stage 3F Fixed-Point Inference Simulation")
    print(f"Project root: {project_root}")

    model = load_model(model_path)
    input_scaler = load_pickle(input_scaler_path)
    output_scaler = load_pickle(output_scaler_path)
    df = load_dataset(dataset_path)

    X_raw = df[FEATURE_COLUMNS].values.astype(np.float32)
    y_actual = df[TARGET_COLUMN].values.astype(np.float32).reshape(-1, 1)

    X_scaled = input_scaler.transform(X_raw).astype(np.float32)

    float_pred_scaled = model.predict(X_scaled, verbose=0)
    float_pred_C = output_scaler.inverse_transform(float_pred_scaled)

    float_params = extract_float_parameters(model)
    fixed_pred_scaled = run_fixed_point_inference_q(X_scaled, float_params)
    fixed_pred_C = output_scaler.inverse_transform(fixed_pred_scaled)

    float_minus_fixed_error = float_pred_C.reshape(-1) - fixed_pred_C.reshape(-1)

    prediction_df = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()
    prediction_df["float_model_Tmax_C"] = float_pred_C.reshape(-1)
    prediction_df["fixed_point_Tmax_C"] = fixed_pred_C.reshape(-1)
    prediction_df["float_minus_fixed_error_C"] = float_minus_fixed_error
    prediction_df["abs_float_minus_fixed_error_C"] = np.abs(float_minus_fixed_error)
    prediction_df["fixed_vs_actual_error_C"] = fixed_pred_C.reshape(-1) - y_actual.reshape(-1)
    prediction_df["abs_fixed_vs_actual_error_C"] = np.abs(prediction_df["fixed_vs_actual_error_C"])
    prediction_df.to_csv(prediction_output_path, index=False)

    metrics = {
        "float_vs_fixed_mae_C": mean_absolute_error(float_pred_C, fixed_pred_C),
        "float_vs_fixed_rmse_C": mean_squared_error(float_pred_C, fixed_pred_C) ** 0.5,
        "float_vs_fixed_max_abs_error_C": float(np.max(np.abs(float_minus_fixed_error))),
        "fixed_vs_actual_mae_C": mean_absolute_error(y_actual, fixed_pred_C),
        "fixed_vs_actual_rmse_C": mean_squared_error(y_actual, fixed_pred_C) ** 0.5,
        "fixed_vs_actual_r2": r2_score(y_actual, fixed_pred_C),
    }

    config_df = save_fixed_point_configuration(parameter_dir)
    save_report(report_path, metrics, config_df)

    plot_float_vs_fixed(prediction_df, figure_dir / "float_vs_fixed_point_predictions.png")
    plot_error_distribution(prediction_df, figure_dir / "fixed_point_error_distribution.png")

    print("Fixed-point inference simulation completed successfully.")
    print(f"Float vs fixed-point MAE: {metrics['float_vs_fixed_mae_C']:.8f} degC")
    print(f"Float vs fixed-point RMSE: {metrics['float_vs_fixed_rmse_C']:.8f} degC")
    print(f"Float vs fixed-point max absolute error: {metrics['float_vs_fixed_max_abs_error_C']:.8f} degC")
    print(f"Fixed-point vs actual COMSOL MAE: {metrics['fixed_vs_actual_mae_C']:.8f} degC")
    print(f"Fixed-point vs actual COMSOL RMSE: {metrics['fixed_vs_actual_rmse_C']:.8f} degC")
    print(f"Fixed-point vs actual COMSOL R2: {metrics['fixed_vs_actual_r2']:.8f}")
    print(f"Prediction file: {prediction_output_path}")
    print(f"Report: {report_path}")
    print(f"Figures: {figure_dir}")
    print(f"Fixed-point parameter folder: {parameter_dir}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print("Execution failed.")
        print(str(exc))
        sys.exit(1)


Stage 3F Fixed-Point Inference Simulation
Project root: /content
Fixed-point inference simulation completed successfully.
Float vs fixed-point MAE: 0.00734069 degC
Float vs fixed-point RMSE: 0.00917451 degC
Float vs fixed-point max absolute error: 0.04298401 degC
Fixed-point vs actual COMSOL MAE: 0.44944221 degC
Fixed-point vs actual COMSOL RMSE: 0.59314358 degC
Fixed-point vs actual COMSOL R2: 0.99947011
Prediction file: /content/dataset/comsol/processed/stage_3F_fixed_point_inference_predictions.csv
Report: /content/reports/stage_3F_fixed_point_inference_report.txt
Figures: /content/figures/stage_3F_fixed_point_inference
Fixed-point parameter folder: /content/models/stage_3F_fixed_point_parameters


In [13]:
from pathlib import Path
import zipfile
from google.colab import files

zip_path = Path("stage_3F_fixed_point_inference_outputs.zip")

output_files = [
    "dataset/comsol/processed/stage_3F_fixed_point_inference_predictions.csv",
    "reports/stage_3F_fixed_point_inference_report.txt",
    "figures/stage_3F_fixed_point_inference/float_vs_fixed_point_predictions.png",
    "figures/stage_3F_fixed_point_inference/fixed_point_error_distribution.png",
    "models/stage_3F_fixed_point_parameters/stage_3F_fixed_point_configuration.txt",
    "models/stage_3F_fixed_point_parameters/stage_3F_fixed_point_scales.csv",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for file in output_files:
        path = Path(file)
        if path.exists():
            z.write(path, arcname=file)
            print("Added:", file)
        else:
            print("Missing:", file)

files.download(str(zip_path))

Added: dataset/comsol/processed/stage_3F_fixed_point_inference_predictions.csv
Added: reports/stage_3F_fixed_point_inference_report.txt
Added: figures/stage_3F_fixed_point_inference/float_vs_fixed_point_predictions.png
Added: figures/stage_3F_fixed_point_inference/fixed_point_error_distribution.png
Added: models/stage_3F_fixed_point_parameters/stage_3F_fixed_point_configuration.txt
Added: models/stage_3F_fixed_point_parameters/stage_3F_fixed_point_scales.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>